# 10 · Uppdatera J18 och Almtunas individuella prestationer
**En notebook för den återkommande uppdateringen.** Kör i Databricks med **Run all**. Ange ditt fungerande API-användarnamn i cell 1; lösenordet läses från `sportlogiq/password`.

1. Kontrollera hela J18:s matchlista och hämta nya eller korrigerade lagunderlag.
2. Bygg Silver, Gold-stjärnschema och passningstäckning i separata arbetsscheman.
3. Hämta individuella mått för alla inlästa Almtuna- och topplagsmatcher, inklusive motståndare i samma matcher. Återanvänd oförändrade svar.
4. Kontrollera nycklar, källversioner och mått. Uppdatera befintliga analystabeller först när arbetsbygget är klart.

Dashboardens sidor, diagram, svenska texter och publiceringsläge bevaras. **Ladda om dashboarden efter SUCCESS.** Ingen dashboardpublicering eller schemaläggning görs här.

### Viktigt inför första körningen
- Notebook 02 och 07 ska redan ha lyckade byggen, och råfilerna från inläsningen ska finnas på volymen. Notebook 04:s resultat-ZIP behövs för de befintliga måttdefinitionerna.
- Denna notebook ersätter den återkommande körningen av 01/02/05/06/07. 03/08/09 behövs bara när presentationen ska ändras. Gamla pilot- och täckningstabeller från 05/06 uppdateras **inte**; dashboardens individuella vyer använder `individual_v1`.
- Kör bara **ett bygge åt gången**, inklusive äldre notebooks. Framtida jobb ska ha högst en samtidig körning.
- Delta-tabeller ersätts var för sig. Dashboarden kan visa blandade eller tomma resultat under det korta slutbytet. Använd resultaten först efter SUCCESS. Versioner registreras före bytet; vid ett fångat fel försöker notebooken återställa ändrade tabeller. Ett avbrutet kluster kan kräva återställning enligt körningsloggen.

### Databetydelse
SOG/xG/istid behåller tidigare kvalitetsregler. Noll och saknat är olika. Positiv verifierad istid styr matchnämnarna för individer. Passningsutfall används inte som prestation när de inte är klassificerade; registrerade passningar bevisar aldrig fullständig tracking. Målvaktsprocent beräknas på summerade skott; noll skott ger NULL.

**Lokalt verifierat:** notebookformat och Python; samtliga 63 Gold-kontroller mot exportfilen i en SQLite-testmiljö; 22 140 individuella mått och 10 avstämningar; cachefel, omstart och simulerad återställning. SQLite-testet verifierar SQL-logiken, inte Spark-kompatibiliteten. Autentiserad API-hämtning, Spark/Delta-skrivning och återställning behöver verifieras genom körningen i din Databricks-miljö. Inga nya fält eller API-rättigheter antas.

## 1. Inställningar

In [0]:
from pathlib import Path
from datetime import datetime, timezone
import json, re, uuid, tempfile, shutil, sys, importlib, hashlib, io, zipfile
from pyspark.sql import functions as F

API_USERNAME = 'oceklund94@gmail.com'  # Ange exakt det användarnamn som fungerade i notebook 07.
SECRET_SCOPE = 'sportlogiq'
CATALOG = 'workspace'
COMPETITION_ID = '300'
SEASON = '20262027'
STAGE = 'regular'
TEAM_ID = '1802'
FETCH_TEAM_IDS = [TEAM_ID, '1804', '1803', '1815', '1866', '1801', '1816', '1805']  # Almtuna + 6 topplag + Frölunda (1805) för individhämtning
PREFIX = 'almtuna_j18_20262027'

MAX_MATCHES_TO_FETCH = 250  # Skydd mot stor oväntad laghämtning; höj vid behov.
FORCE_GAME_IDS = []  # Exempel ['224914']: hämta om även när API:ets processmarkör är oförändrad.
REFRESH_API = True  # False bygger om från befintligt lagunderlag; saknade spelaruppgifter hämtas ändå.

for value in (CATALOG, PREFIX):
    assert re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*', value)
for value in (COMPETITION_ID, SEASON, TEAM_ID):
    assert value.isdigit()
assert STAGE == 'regular' and PREFIX == f'almtuna_j18_{SEASON}'
assert isinstance(MAX_MATCHES_TO_FETCH, int) and MAX_MATCHES_TO_FETCH > 0
FORCED = {str(g) for g in FORCE_GAME_IDS}
assert all(g.isdigit() for g in FORCED)
assert REFRESH_API or not FORCED, 'FORCE_GAME_IDS kräver REFRESH_API=True.'
RUN_ID = uuid.uuid4().hex
RUN_TIME = datetime.now(timezone.utc).isoformat()
VOLUME = Path(f'/Volumes/{CATALOG}/almtuna_assets/files')
RAW = VOLUME/PREFIX/'raw'
CACHE = VOLUME/PREFIX/'refresh10_cache'
RUN_DIR = VOLUME/PREFIX/'refresh10_runs'/RUN_ID
assert RAW.is_dir(), f'Råarkivet saknas: {RAW}'
RUN_DIR.mkdir(parents=True, exist_ok=True)
PROD_BRONZE = f'{CATALOG}.{PREFIX}_bronze'
PROD_SILVER = f'{CATALOG}.{PREFIX}_silver'
PROD_BASE = f'{CATALOG}.{PREFIX}_gold_star_v1'
PROD_INDIVIDUAL = f'{CATALOG}.{PREFIX}_individual_v1'
PROD_PASSING = f'{CATALOG}.{PREFIX}_passing_v1'
WORK_BRONZE = f'{CATALOG}.{PREFIX}_refresh10_bronze'
WORK_SILVER = f'{CATALOG}.{PREFIX}_refresh10_silver'
WORK_BASE = f'{CATALOG}.{PREFIX}_refresh10_gold_star_v1'
WORK_INDIVIDUAL = f'{CATALOG}.{PREFIX}_refresh10_individual_v1'
WORK_PASSING = f'{CATALOG}.{PREFIX}_refresh10_passing_v1'
print('Körning:', RUN_ID)
print('Lagrådata:', RAW)
print('Omfattning: hela J18 för lagdata; Almtuna + topplag för individer.')

## 2. Läs in den medföljande byggkoden
Notebooken innehåller parsern och kontrollerna från de fungerande versionerna. Originalpaketet behöver inte laddas upp igen.

In [0]:
EMBEDDED_FILES = {'aj18_refresh10/__init__.py': '', 'aj18_refresh10/api.py': 'import argparse, csv, getpass, http.cookiejar, json, os, re, sys, time, urllib.request, urllib.error, urllib.parse, zipfile\nfrom pathlib import Path\nfrom datetime import datetime, timezone\n\nSECRET_KEYS = re.compile(r\'token|password|passwd|authorization|cookie|secret|api.?key|username|email\', re.I)\nSECRETS = []\n\ndef clean(value):\n    if isinstance(value, dict):\n        return {k: \'[REDACTED]\' if SECRET_KEYS.search(k) else clean(v) for k,v in value.items()}\n    if isinstance(value, list): return [clean(v) for v in value]\n    if isinstance(value, str):\n        if value.startswith((\'https://\',\'http://\')):\n            parsed = urllib.parse.urlsplit(value)\n            if parsed.query or parsed.fragment:\n                value = urllib.parse.urlunsplit((parsed.scheme,parsed.netloc,parsed.path,\'\',\'\')) + \'?[REDACTED]\'\n        for secret in sorted(set(SECRETS), key=len, reverse=True):\n            if secret: value = value.replace(secret, \'[REDACTED]\')\n    return value\n\ndef fields(value):\n    result = {}\n    def walk(v, path):\n        row = result.setdefault(path, {\'occurrences\':0,\'non_null\':0,\'types\':set()})\n        row[\'occurrences\'] += 1\n        row[\'non_null\'] += v is not None\n        row[\'types\'].add(type(v).__name__)\n        if isinstance(v, dict):\n            for k, val in v.items(): walk(val, path+\'.\'+k)\n        elif isinstance(v, list):\n            for val in v: walk(val, path+\'[]\')\n    walk(value, \'$\')\n    return [{\'field\':p,\'occurrences\':r[\'occurrences\'],\'non_null\':r[\'non_null\'],\'types\':\',\'.join(sorted(r[\'types\']))} for p,r in sorted(result.items())]\n\ndef config(folder):\n    cfg = {}\n    for p in dict.fromkeys([Path.cwd()/\'.env\', Path(folder)/\'.env\']):\n        if p.is_file():\n            for line in p.read_text(encoding=\'utf-8-sig\').splitlines():\n                m = re.match(r\'^\\s*(?:export\\s+)?(SPORTLOGIQ_[A-Z_]+)\\s*=\\s*(.*?)\\s*$\',line)\n                if m:\n                    value=m[2]\n                    if len(value)>1 and value[0]==value[-1] and value[0] in "\\"\'": value=value[1:-1]\n                    cfg[m[1]]=value\n    cfg.update({k:v for k,v in os.environ.items() if k.startswith(\'SPORTLOGIQ_\')})\n    return cfg\n\nclass NoRedirect(urllib.request.HTTPRedirectHandler):\n    def redirect_request(self, req, fp, code, msg, headers, newurl): return None\n\nclass Client:\n    def __init__(self, base):\n        self.base=base.rstrip(\'/\')\n        self.opener=urllib.request.build_opener(NoRedirect(),urllib.request.HTTPCookieProcessor(http.cookiejar.CookieJar()))\n        self.auth={}\n    def call(self, path, params=None, login=None):\n        url=self.base+path\n        if params: url+=\'?\'+urllib.parse.urlencode(params,doseq=True)\n        hdr={\'Accept\':\'application/json\',\'User-Agent\':\'almtuna-api-check/1.0\',**self.auth}\n        data=None\n        if login is not None:\n            hdr={\'Accept\':\'application/json\',\'Content-Type\':\'application/json\'}\n            data=json.dumps(login).encode()\n        for attempt in range(3):\n            try:\n                with self.opener.open(urllib.request.Request(url,data=data,headers=hdr),timeout=45) as res:\n                    status=res.status\n                    raw=res.read()\n                try: payload=json.loads(raw)\n                except (ValueError,UnicodeDecodeError): return status,None,\'non_json_response\'\n                return status,payload,\'ok\'\n            except urllib.error.HTTPError as e:\n                status=e.code\n                e.close()\n                if (status==429 or status>=500) and attempt<2:\n                    time.sleep(2*(attempt+1)); continue\n                return status,None,\'http_error\'\n            except (OSError,TimeoutError): return None,None,\'connection_error\'\n\ndef topic_ids(payload):\n    if isinstance(payload,list): items=payload\n    elif isinstance(payload,dict): items=payload.get(\'topics\',[])\n    else: items=[]\n    return [str(x[\'id\']) for x in items if isinstance(x,dict) and x.get(\'id\') is not None]\n\ndef authenticate(c,cfg,allow_prompt=True):\n    token=cfg.get(\'SPORTLOGIQ_TOKEN\',\'\')\n    username=cfg.get(\'SPORTLOGIQ_USERNAME\',\'\')\n    password=cfg.get(\'SPORTLOGIQ_PASSWORD\',\'\')\n    SECRETS.extend([token,username,password])\n    def detect(tok):\n        variants=[]\n        header=cfg.get(\'SPORTLOGIQ_AUTH_HEADER\',\'auto\')\n        if header.lower()!=\'auto\':\n            prefix=cfg.get(\'SPORTLOGIQ_AUTH_PREFIX\',\'Bearer\').strip()\n            variants.append({header:(prefix+\' \' if prefix else \'\')+tok})\n        variants += [{\'token\':tok},{\'Authorization\':tok},{\'Authorization\':\'Bearer \'+tok},{\'X-Access-Token\':tok},{\'X-Auth-Token\':tok},{}] if tok else [{}]\n        last=None\n        for headers in variants:\n            c.auth=headers\n            status,body,kind=c.call(\'/v1/hockey/competitions\')\n            last=status\n            if status==200 and kind==\'ok\': return True\n            if status not in (401,403): break\n        print(\'Authentication check HTTP:\',last if last is not None else \'connection error\')\n        return False\n    if token and detect(token): return True\n    if token: print(\'Existing token could not be verified. Trying username/password login.\')\n    if not allow_prompt and (not username or not password): return False\n    if not username: username=input(\'Sportlogiq username: \').strip()\n    if not password: password=getpass.getpass(\'Sportlogiq password (hidden): \')\n    SECRETS.extend([username,password])\n    status,body,kind=c.call(\'/v1/hockey/login\',login={\'username\':username,\'password\':password})\n    if status not in (200,201) or kind!=\'ok\':\n        print(\'Login failed. HTTP:\',status,\'No credentials were saved.\'); return False\n    token=body.get(\'token\',\'\') if isinstance(body,dict) else \'\'\n    SECRETS.append(token)\n    return detect(token)\n', 'aj18_refresh10/model.py': '"""Sportlogiq J18 model. All rows retain source snapshot and match scope."""\nfrom collections import defaultdict, Counter\nfrom pathlib import Path\nimport csv, gzip, hashlib, json, math, sqlite3\n\n# SQL-neutral types. Spark conversion is in databricks.py.\nSCHEMAS = {\n \'games\': {\'game_id\':\'TEXT\',\'competition_id\':\'TEXT\',\'season\':\'TEXT\',\'stage\':\'TEXT\',\'game_date\':\'TEXT\',\'home_team_id\':\'TEXT\',\'away_team_id\':\'TEXT\',\'home_score\':\'INTEGER\',\'away_score\':\'INTEGER\',\'status\':\'TEXT\',\'period_count\':\'INTEGER\',\'source_snapshot\':\'TEXT\'},\n \'teams\': {\'team_id\':\'TEXT\',\'team_name\':\'TEXT\',\'team_shorthand\':\'TEXT\'},\n \'players\': {\'player_id\':\'TEXT\',\'player_name\':\'TEXT\'},\n \'rosters\': {\'game_id\':\'TEXT\',\'team_id\':\'TEXT\',\'player_id\':\'TEXT\',\'jersey\':\'TEXT\',\'position\':\'TEXT\',\'source_snapshot\':\'TEXT\'},\n \'events\': {\'event_key\':\'TEXT\',\'game_id\':\'TEXT\',\'event_id\':\'TEXT\',\'base_event_id\':\'TEXT\',\'team_id\':\'TEXT\',\'player_id\':\'TEXT\',\'period\':\'INTEGER\',\'period_time\':\'REAL\',\'game_time\':\'REAL\',\'event_name\':\'TEXT\',\'event_type\':\'TEXT\',\'outcome\':\'TEXT\',\'zone\':\'TEXT\',\'manpower\':\'TEXT\',\'team_skaters\':\'INTEGER\',\'opponent_skaters\':\'INTEGER\',\'x_adj\':\'REAL\',\'y_adj\':\'REAL\',\'possession_id\':\'TEXT\',\'flags_json\':\'TEXT\',\'is_shot_attempt\':\'INTEGER\',\'is_sog\':\'INTEGER\',\'is_goal\':\'INTEGER\',\'is_shootout\':\'INTEGER\',\'source_snapshot\':\'TEXT\'},\n \'shots\': {\'shot_key\':\'TEXT\',\'game_id\':\'TEXT\',\'team_id\':\'TEXT\',\'player_id\':\'TEXT\',\'position\':\'TEXT\',\'period\':\'INTEGER\',\'period_time\':\'REAL\',\'game_time\':\'REAL\',\'frame\':\'INTEGER\',\'zone\':\'TEXT\',\'play_section\':\'TEXT\',\'shot_type\':\'TEXT\',\'outcome\':\'TEXT\',\'manpower\':\'TEXT\',\'team_skaters\':\'INTEGER\',\'opponent_skaters\':\'INTEGER\',\'x_adj\':\'REAL\',\'y_adj\':\'REAL\',\'is_sog\':\'INTEGER\',\'is_goal\':\'INTEGER\',\'is_slot\':\'INTEGER\',\'is_inner_slot\':\'INTEGER\',\'is_shootout\':\'INTEGER\',\'xg_on_net\':\'REAL\',\'xg_all_shots\':\'REAL\',\'source_snapshot\':\'TEXT\'},\n \'toi_events\': {\'toi_event_key\':\'TEXT\',\'game_id\':\'TEXT\',\'team_id\':\'TEXT\',\'player_id\':\'TEXT\',\'period\':\'INTEGER\',\'period_time\':\'REAL\',\'game_time\':\'REAL\',\'in_or_out\':\'TEXT\',\'manpower\':\'TEXT\',\'team_skaters\':\'INTEGER\',\'opponent_skaters\':\'INTEGER\',\'source_snapshot\':\'TEXT\'},\n \'shifts\': {\'shift_key\':\'TEXT\',\'game_id\':\'TEXT\',\'team_id\':\'TEXT\',\'player_id\':\'TEXT\',\'period\':\'INTEGER\',\'start_period_sec\':\'REAL\',\'end_period_sec\':\'REAL\',\'duration_sec\':\'REAL\',\'source_snapshot\':\'TEXT\'},\n \'metrics\': {\'metric_key_id\':\'TEXT\',\'game_id\':\'TEXT\',\'team_id\':\'TEXT\',\'player_id\':\'TEXT\',\'period\':\'INTEGER\',\'topic_id\':\'TEXT\',\'breakdown_json\':\'TEXT\',\'metric_name\':\'TEXT\',\'metric_value\':\'REAL\',\'query_json\':\'TEXT\',\'source_snapshot\':\'TEXT\'},\n \'team_metrics\': {\'game_id\':\'TEXT\',\'team_id\':\'TEXT\',\'metric_name\':\'TEXT\',\'metric_value\':\'REAL\',\'topic_ids\':\'TEXT\',\'source_snapshot\':\'TEXT\'},\n \'quality\': {\'game_id\':\'TEXT\',\'check_name\':\'TEXT\',\'severity\':\'TEXT\',\'detail\':\'TEXT\',\'source_snapshot\':\'TEXT\'},\n \'coverage\': {\'game_id\':\'TEXT\',\'endpoint\':\'TEXT\',\'http_status\':\'INTEGER\',\'result\':\'TEXT\',\'source_snapshot\':\'TEXT\'}\n}\nKEYS = {\'games\':[\'game_id\'],\'teams\':[\'team_id\'],\'players\':[\'player_id\'],\'rosters\':[\'game_id\',\'team_id\',\'player_id\'], \'events\':[\'event_key\'],\'shots\':[\'shot_key\'],\'toi_events\':[\'toi_event_key\'],\'shifts\':[\'shift_key\'],\'metrics\':[\'metric_key_id\'],\'team_metrics\':[\'game_id\',\'team_id\',\'metric_name\']}\n\ndef ident(v):\n    return str(v).strip() if v is not None and str(v).strip() not in (\'\',\'None\',\'null\') else None\n\ndef number(v):\n    try:\n        n=float(v)\n        return n if math.isfinite(n) else None\n    except (ValueError,TypeError):return None\n\ndef key(*parts):return hashlib.sha256(json.dumps(parts,ensure_ascii=False,sort_keys=True).encode()).hexdigest()\ndef events_of(payload):return payload.get(\'events\',[]) if isinstance(payload,dict) else payload if isinstance(payload,list) else []\ndef safe_name(team):\n    location=str(team.get(\'location\') or \'\').strip(); name=str(team.get(\'name\') or \'\').strip()\n    return name if not location or location.casefold() in name.casefold() else location+\' \'+name\n\ndef normalize(bundle):\n    tables={name:[] for name in SCHEMAS}\n    source=bundle[\'snapshot_id\']; gid=ident(bundle[\'game\'][\'id\']); comp=ident(bundle[\'competition_id\']); game=bundle[\'game\']\n    def issue(name,detail,severity=\'warning\'):\n        tables[\'quality\'].append(dict(game_id=gid,check_name=name,severity=severity,detail=detail,source_snapshot=source))\n    def add(table,**row):\n        if \'source_snapshot\' in SCHEMAS[table]:row.setdefault(\'source_snapshot\',source)\n        tables[table].append({k:row.get(k) for k in SCHEMAS[table]})\n    sides={}\n    aliases={}\n    for side in (\'home\',\'away\'):\n        team=game.get(side+\'_team\') or {}; tid=ident(team.get(\'id\') or game.get(side+\'_team_id\')); sides[side]=tid\n        add(\'teams\',team_id=tid,team_name=safe_name(team),team_shorthand=team.get(\'shorthand\'))\n        for a in (tid,team.get(\'name\'),team.get(\'shorthand\'),safe_name(team)):\n            if a:aliases.setdefault(str(a).strip(),set()).add(tid)\n    roster=bundle.get(\'roster\') or {}; player_teams={}\n    for tid in sides.values():\n        for p in (roster.get(tid) or {}).get(\'players\',[]):\n            pid=ident(p.get(\'id\'));player_teams.setdefault(pid,set()).add(tid)\n            add(\'players\',player_id=pid,player_name=\' \'.join(str(p.get(k) or \'\').strip() for k in (\'first_name\',\'last_name\')).strip())\n            add(\'rosters\',game_id=gid,team_id=tid,player_id=pid,jersey=ident(p.get(\'jersey_number\')),position=p.get(\'position\'))\n    def resolve_team(e):\n        if ident(e.get(\'team_id\')) in sides.values():return ident(e[\'team_id\'])\n        a=aliases.get(str(e.get(\'team\') or \'\').strip(),set())\n        if len(a)==1:return next(iter(a))\n        a=player_teams.get(ident(e.get(\'player_reference_id\') or e.get(\'player_id\')),set())\n        return next(iter(a)) if len(a)==1 else None\n    periods={int(v[\'period\']):v.get(\'period_type\') for v in game.get(\'videos\',[]) if v.get(\'period\') is not None}\n    def shootout(e):return int(periods.get(int(e.get(\'period\') or 0))==\'shootout\')\n    add(\'games\',game_id=gid,competition_id=comp,season=str(game.get(\'season\') or bundle[\'season\']),stage=game.get(\'stage\') or bundle[\'stage\'],game_date=game.get(\'date\'),home_team_id=sides[\'home\'],away_team_id=sides[\'away\'],home_score=game.get(\'home_final_score\'),away_score=game.get(\'away_final_score\'),status=game.get(\'event_status\'),period_count=game.get(\'period_count\'))\n    for ep,c in bundle.get(\'coverage\',{}).items():\n        add(\'coverage\',game_id=gid,endpoint=ep,http_status=c.get(\'http_status\'),result=c.get(\'result\'))\n        if c.get(\'http_status\')!=200:issue(\'api_\'+ep,\'HTTP \'+str(c.get(\'http_status\'))+\': \'+str(c.get(\'result\')))\n    for e in events_of(bundle.get(\'compiled\')):\n        eid=ident(e.get(\'event_id\'));tid=resolve_team(e);flags=e.get(\'flags\') or []\n        if eid is None:issue(\'missing_event_id\',\'Compiled event without event_id\',\'error\');continue\n        shot=e.get(\'name\')==\'shot\'\n        add(\'events\',event_key=key(comp,gid,eid),game_id=gid,event_id=eid,base_event_id=ident(e.get(\'base_event_id\')),team_id=tid,player_id=ident(e.get(\'player_id\')),period=e.get(\'period\'),period_time=number(e.get(\'period_time\')),game_time=number(e.get(\'game_time\')),event_name=e.get(\'name\'),event_type=e.get(\'type\'),outcome=e.get(\'outcome\'),zone=e.get(\'zone\'),manpower=e.get(\'manpowersituation\'),team_skaters=e.get(\'team_skaters_on_ice\'),opponent_skaters=e.get(\'opposing_team_skaters_on_ice\'),x_adj=number(e.get(\'x_adj_coord\')),y_adj=number(e.get(\'y_adj_coord\')),possession_id=ident(e.get(\'current_possession\')),flags_json=json.dumps(flags),is_shot_attempt=int(shot),is_sog=int(shot and e.get(\'outcome\')==\'successful\'),is_goal=int(e.get(\'name\')==\'goal\' and e.get(\'outcome\')==\'successful\'),is_shootout=shootout(e))\n    for i,e in enumerate(events_of(bundle.get(\'full\'))):\n        if e.get(\'name\')!=\'shot\':continue\n        tid=resolve_team(e); pid=ident(e.get(\'player_reference_id\'));flags=e.get(\'flags\') or []\n        add(\'shots\',shot_key=key(comp,gid,\'full\',i),game_id=gid,team_id=tid,player_id=pid,position=e.get(\'player_position\'),period=e.get(\'period\'),period_time=number(e.get(\'period_time\')),game_time=number(e.get(\'game_time\')),frame=e.get(\'frame\'),zone=e.get(\'zone\'),play_section=e.get(\'play_section\'),shot_type=e.get(\'type\'),outcome=e.get(\'outcome\'),manpower=e.get(\'manpower_situation\'),team_skaters=e.get(\'team_skaters_on_ice\'),opponent_skaters=e.get(\'opposing_team_skaters_on_ice\'),x_adj=number(e.get(\'x_adj_coord\')),y_adj=number(e.get(\'y_adj_coord\')),is_sog=int(e.get(\'outcome\')==\'successful\'),is_goal=int(\'withgoal\' in flags),is_slot=int(str(e.get(\'type\') or \'\').startswith(\'slot\')),is_inner_slot=int(e.get(\'play_section\')==\'innerSlot\'),is_shootout=shootout(e),xg_on_net=number(e.get(\'expected_goals_on_net\')),xg_all_shots=number(e.get(\'expected_goals_all_shots\')))\n    # IN/OUT pairing within player and period. Never invent a missing boundary.\n    grouped=defaultdict(list)\n    for i,e in enumerate(events_of(bundle.get(\'toi\'))):\n        tid=resolve_team(e);pid=ident(e.get(\'player_reference_id\'));t=number(e.get(\'period_time\'));state=str(e.get(\'in_or_out\') or \'\').upper()\n        add(\'toi_events\',toi_event_key=key(comp,gid,\'toi\',i),game_id=gid,team_id=tid,player_id=pid,period=e.get(\'period\'),period_time=t,game_time=number(e.get(\'game_time\')),in_or_out=state,manpower=e.get(\'manpower_situation\'),team_skaters=e.get(\'team_skaters_on_ice\'),opponent_skaters=e.get(\'opposing_team_skaters_on_ice\'))\n        if tid and pid and t is not None and e.get(\'period\') is not None:grouped[(tid,pid,e[\'period\'])].append((t,i,state))\n        else:issue(\'toi_identity\',\'Unresolved player/team/time in TOI event\')\n    for (tid,pid,period),es in grouped.items():\n        opened=None\n        for t,i,state in sorted(es):\n            if state==\'IN\':\n                if opened is not None:issue(\'unpaired_in\',f\'{pid}, P{period}, {opened[0]} -> {t}\')\n                opened=(t,i)\n            elif state==\'OUT\':\n                if opened is None:issue(\'unpaired_out\',f\'{pid}, P{period}, {t}\')\n                else:\n                    start,idx=opened;opened=None\n                    if t>start:add(\'shifts\',shift_key=key(comp,gid,pid,period,idx),game_id=gid,team_id=tid,player_id=pid,period=period,start_period_sec=start,end_period_sec=t,duration_sec=t-start)\n                    elif t<start:issue(\'negative_shift\',f\'{pid}, P{period}\', \'error\')\n            else:issue(\'unknown_shift_state\',str(state))\n        if opened is not None:issue(\'open_shift\',f\'{pid}, P{period}, {opened[0]}\')\n    if 0<len(tables[\'toi_events\'])<100:issue(\'sparse_toi\',f"Only {len(tables[\'toi_events\'])} TOI rows; inspect before comparing ice time.")\n    if not tables[\'toi_events\']:issue(\'empty_toi\',\'No playerTOI events\')\n    # Keep every topic; deduplicate exact team-level metric names only after conflict check.\n    candidates=defaultdict(list)\n    for obj in bundle.get(\'metric_payloads\',[]):\n        payload=obj.get(\'payload\') or {};topic=ident(payload.get(\'topic_id\') or obj.get(\'topic_id\'));query=obj.get(\'query\') or {}\n        for grouping in payload.get(\'groupings\',[]):\n            breakdown=grouping.get(\'breakdown\') or [];bd={x[\'key\']:x.get(\'value\') for x in breakdown};bj=json.dumps(breakdown,sort_keys=True,ensure_ascii=False)\n            tid=ident(bd.get(\'team\'));pid=ident(bd.get(\'player\'));period=number(bd.get(\'period\'))\n            for metric in grouping.get(\'metrics\',[]):\n                name=metric.get(\'key\');value=number(metric.get(\'value\'))\n                add(\'metrics\',metric_key_id=key(comp,gid,topic,bj,name,json.dumps(query,sort_keys=True)),game_id=gid,team_id=tid,player_id=pid,period=int(period) if period is not None else None,topic_id=topic,breakdown_json=bj,metric_name=name,metric_value=value,query_json=json.dumps(query,sort_keys=True))\n                if set(bd)=={\'team\'} and tid and name and not any(k in query for k in (\'manpowersituation\',\'skatersonice\',\'positions\',\'period\',\'game_time_start\',\'game_time_end\')):candidates[(tid,name)].append((topic,value))\n    for (tid,name),items in candidates.items():\n        values={v for _,v in items if v is not None}\n        value=next(iter(values)) if len(values)==1 else None\n        if len(values)>1:issue(\'metric_conflict\',f\'{tid}: {name}, different values across topics\',\'error\')\n        add(\'team_metrics\',game_id=gid,team_id=tid,metric_name=name,metric_value=value,topic_ids=\',\'.join(sorted({t for t,_ in items if t})))\n    # Cross-source reconciliations: failures are visible and never silently overwritten.\n    lookup={(r[\'team_id\'],r[\'metric_name\']):r[\'metric_value\'] for r in tables[\'team_metrics\']}\n    checks=[(\'sog\',\'Total Successful Shots Taken\',lambda s:s[\'is_sog\']),(\'slot_sog\',\'Total Successful Shot From Slot Attempts\',lambda s:s[\'is_sog\']*s[\'is_slot\']),(\'inner_slot_sog\',\'Total Successful Shots Taken From Inner Slot\',lambda s:s[\'is_sog\']*s[\'is_inner_slot\']),(\'goals\',\'Total Regular Goals\',lambda s:s[\'is_goal\'])]\n    for tid in sides.values():\n        shots=[s for s in tables[\'shots\'] if s[\'team_id\']==tid and not s[\'is_shootout\']]\n        for name,metric,fn in checks:\n            expected=lookup.get((tid,metric));actual=sum(fn(s) for s in shots)\n            if expected is not None and actual!=expected:issue(\'reconcile_\'+name,f\'{tid}: shot feed={actual}, API metric={expected}\',\'error\')\n        ce=[e for e in tables[\'events\'] if e[\'team_id\']==tid and not e[\'is_shootout\']]\n        for name,col in [(\'sog\',\'is_sog\'),(\'goals\',\'is_goal\')]:\n            a=sum(e[col] for e in ce);b=sum(s[col] for s in shots)\n            if a!=b:issue(\'feeds_\'+name,f\'{tid}: compiled={a}, full shots={b}\',\'error\')\n    if any(s[\'team_id\'] is None for s in tables[\'shots\']):issue(\'unresolved_shot_team\',\'At least one shot cannot be assigned to a team\',\'error\')\n    # Enforce uniqueness rather than declaring unchecked RELY constraints.\n    for table,cols in KEYS.items():\n        seen={}; unique=[]\n        for r in tables[table]:\n            k=tuple(r[c] for c in cols)\n            if k in seen:\n                if r!=seen[k]:issue(\'duplicate_\'+table,str(k),\'error\')\n                continue\n            seen[k]=r;unique.append(r)\n        tables[table]=unique\n    if not tables[\'quality\']:issue(\'reconciliation\',\'Shot feeds and API metrics agree; IN/OUT pairs and primary keys checked.\',\'info\')\n    return tables\n\n\ndef read_bundle(path):\n    with gzip.open(path,\'rt\',encoding=\'utf-8\') as f:return json.load(f)\n\ndef materialize(bundles,output):\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    all_tables={n:[] for n in SCHEMAS}\n    for bundle in bundles:\n        for n,rows in normalize(bundle).items():all_tables[n].extend(rows)\n    # Dimensions are current snapshots. Roster is the historical team/position authority.\n    for table in (\'teams\',\'players\'):\n        col=KEYS[table][0];all_tables[table]=list({r[col]:r for r in all_tables[table]}.values())\n    for name,rows in all_tables.items():\n        with (output/(name+\'.jsonl\')).open(\'w\',encoding=\'utf-8\') as f:\n            for row in rows:f.write(json.dumps(row,ensure_ascii=False)+\'\\n\')\n        with (output/(name+\'.csv\')).open(\'w\',encoding=\'utf-8-sig\',newline=\'\') as f:\n            w=csv.DictWriter(f,fieldnames=list(SCHEMAS[name]));w.writeheader();w.writerows(rows)\n    db=output/\'almtuna.sqlite\'\n    with sqlite3.connect(db) as conn:\n        for name,cols in SCHEMAS.items():\n            conn.execute(\'DROP TABLE IF EXISTS \'+name)\n            conn.execute(\'CREATE TABLE \'+name+\' (\'+\',\'.join(k+\' \'+v for k,v in cols.items())+\')\')\n            conn.executemany(\'INSERT INTO \'+name+\' VALUES (\'+\',\'.join(\'?\' for _ in cols)+\')\',[[r[k] for k in cols] for r in all_tables[name]])\n        sql=(Path(__file__).resolve().parents[1]/\'sql/gold.sql\').read_text(encoding=\'utf-8\')\n        conn.executescript(sql)\n    return all_tables\n', 'aj18_refresh10/ingestion.py': "from pathlib import Path\nimport json, gzip, hashlib, uuid\nfrom datetime import datetime, timezone\n\n\ndef digest(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False, allow_nan=False).encode()).hexdigest()\n\n\ndef current_raw(raw, competition, season, stage, reader):\n    folder = Path(raw)/competition/season/stage\n    found = {}\n    if folder.exists():\n        for directory in sorted(folder.iterdir()):\n            if not directory.is_dir():\n                continue\n            files = sorted(directory.glob('*.json.gz'))\n            if files:\n                bundle = reader(files[-1])\n                gid = str(bundle['game']['id'])\n                if gid != directory.name or gid in found:\n                    raise ValueError('Fel/dubbel match i råarkivet: '+directory.name)\n                if (str(bundle['competition_id']), str(bundle['season']), bundle['stage']) != (competition, season, stage):\n                    raise ValueError('Fel tävling/säsong i råarkivet: '+gid)\n                found[gid] = (files[-1], bundle)\n    return found\n\n\ndef fetch_json(client, path, params=None):\n    status, body, kind = client.call(path, params)\n    if status != 200 or kind != 'ok' or body is None:\n        raise RuntimeError(f'API-anrop misslyckades: {path}, HTTP {status}, {kind}. Sparade svar kan återanvändas vid omkörning.')\n    return body\n\n\ndef list_games(client, competition, season, stage):\n    body = fetch_json(client, '/v1/hockey/games', {'competition_id': competition, 'season': season, 'stage': stage, 'include_upcoming': 0})\n    if not isinstance(body, dict) or not isinstance(body.get('games'), list):\n        raise ValueError('Matchlistans format har ändrats.')\n    if str(body.get('competition_id', competition)) != competition:\n        raise ValueError('Fel tävling i matchlistan.')\n    games = body['games']\n    ids = [str(g['id']) for g in games]\n    if len(ids) != len(set(ids)) or any(not x.isdigit() for x in ids):\n        raise ValueError('Matchlistan innehåller felaktiga/dubbla ID.')\n    return {str(g['id']): g for g in games}\n\n\ndef needs_fetch(game, existing, topics, forced=False):\n    if forced or existing is None:\n        return True\n    marker = game.get('last_metrics_full_process_time')\n    if not marker or existing.get('last_metrics_full_process_time') != marker:\n        return True\n    required = {'game', 'roster', 'compiled', 'full', 'toi', 'shift_events'} | {'metric_'+t for t in topics}\n    coverage = existing.get('coverage', {})\n    return not required.issubset(coverage) or any(coverage[k].get('http_status') != 200 or coverage[k].get('result') != 'ok' for k in required)\n\n\ndef cached_fetch(client, path, params, file, revision, clean):\n    entry = None\n    if file.exists():\n        try:\n            item = json.loads(file.read_text(encoding='utf-8'))\n            if (item.get('revision') == revision and item.get('endpoint') == path\n                    and item.get('params') == params and isinstance(item.get('response'), (dict, list))\n                    and item.get('response_sha256') == digest(item['response'])):\n                entry = item\n        except (ValueError, OSError, KeyError):\n            pass\n    if entry is None:\n        response = clean(fetch_json(client, path, params))\n        entry = {'revision': revision, 'endpoint': path, 'params': params,\n                 'retrieved_at': datetime.now(timezone.utc).isoformat(), 'response': response,\n                 'response_sha256': digest(response)}\n        file.parent.mkdir(parents=True, exist_ok=True)\n        # Full immutable payload per endpoint. An interrupted file is rejected on the next run.\n        file.write_text(json.dumps(entry, ensure_ascii=False, allow_nan=False), encoding='utf-8')\n    return entry\n\n\ndef validate_detail(game, gid, competition, season, stage):\n    if not isinstance(game, dict) or str(game.get('id')) != gid:\n        raise ValueError('Fel matchdetalj: '+gid)\n    if str(game.get('season')) != season or game.get('stage') != stage:\n        raise ValueError('Fel säsong/steg: '+gid)\n    if game.get('competition_id') is not None and str(game['competition_id']) != competition:\n        raise ValueError('Fel tävling: '+gid)\n    sides = [str((game.get(s+'_team') or {}).get('id', '')) for s in ('home', 'away')]\n    if len(set(sides)) != 2 or any(not s.isdigit() for s in sides):\n        raise ValueError('Ogiltiga lag: '+gid)\n\n\ndef sync_selected(client, games, raw_index, topics, candidates, raw, cache, competition, season, stage, run_id, forced, clean):\n    results = []\n    for index, gid in enumerate(candidates, 1):\n        game = fetch_json(client, '/v1/hockey/games/'+gid)\n        validate_detail(game, gid, competition, season, stage)\n        if game.get('event_status') != 'over':\n            results.append({'game_id': gid, 'status': 'not_finished'})\n            continue\n        marker = games[gid].get('last_metrics_full_process_time')\n        # No process marker => conservative fresh fetch. Forced corrections also bypass old payloads.\n        revision = digest({'game': game, 'marker': marker, 'topics': topics,\n                           'fresh_run': run_id if not marker or gid in forced else None})\n        root = cache/'team'/gid/revision\n        bundle = {'snapshot_id': datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'),\n                  'competition_id': competition, 'season': season, 'stage': stage, 'game': clean(game),\n                  'coverage': {'game': {'http_status': 200, 'result': 'ok'}},\n                  'last_metrics_full_process_time': marker, 'metric_payloads': []}\n        for alias, suffix in [('roster','roster'), ('compiled','events/compiled'), ('full','events/full'), ('toi','playerTOI'), ('shift_events','events/shifts')]:\n            entry = cached_fetch(client, '/v1/hockey/games/'+gid+'/'+suffix, None, root/(alias+'.json'), revision, clean)\n            bundle[alias] = entry['response']\n            bundle['coverage'][alias] = {'http_status': 200, 'result': 'ok'}\n        for topic in topics:\n            params = {'breakdown': ['team']}\n            entry = cached_fetch(client, '/v1/hockey/games/'+gid+'/metrics/'+topic, params, root/('metric_'+topic+'.json'), revision, clean)\n            payload = entry['response']\n            if not isinstance(payload, dict) or str(payload.get('id')) != gid or str(payload.get('topic_id')) != topic or not isinstance(payload.get('groupings'), list):\n                raise ValueError('Fel format i lagmått: '+gid+'/'+topic)\n            bundle['metric_payloads'].append({'topic_id': topic, 'query': params, 'payload': payload})\n            bundle['coverage']['metric_'+topic] = {'http_status': 200, 'result': 'ok'}\n        target = raw/competition/season/stage/gid/(bundle['snapshot_id']+'.json.gz')\n        target.parent.mkdir(parents=True, exist_ok=True)\n        compressed = gzip.compress(json.dumps(bundle, ensure_ascii=False, allow_nan=False).encode())\n        # Only publish complete raw bundles. A partial .pending is not selected by current_raw.\n        pending = target.with_suffix('.pending')\n        pending.write_bytes(compressed)\n        pending.replace(target)\n        raw_index[gid] = (target, bundle)\n        results.append({'game_id': gid, 'status': 'saved'})\n        print(f'{index}/{len(candidates)} lagunderlag klart: {gid}', flush=True)\n    return results\n\n\ndef raw_revision(path):\n    return hashlib.sha256(path.read_bytes()).hexdigest()\n\n\ndef legacy_entries(gid, snapshot, production_snapshots, legacy_manifest):\n    if production_snapshots.get(gid) != snapshot:\n        return {}\n    entries = {}\n    for row in legacy_manifest:\n        if row['game_key'] != gid:\n            continue\n        path = Path(row['path'])\n        try:\n            raw = path.read_bytes()\n            if hashlib.sha256(raw).hexdigest() != row['sha256']:\n                continue\n            entry = json.loads(raw)\n            if isinstance(entry.get('response'), (dict, list)):\n                entries[row['topic']] = entry\n        except (OSError, ValueError):\n            continue\n    return entries\n", 'sql/gold.sql': "-- Portable SQL: executed locally in SQLite and as Spark SQL views in Databricks.\nDROP VIEW IF EXISTS v_team_game;\nCREATE VIEW v_team_game AS\nWITH sides AS (\n SELECT game_id,home_team_id AS team_id,away_team_id AS opponent_id,home_score AS goals,away_score AS goals_against FROM games\n UNION ALL\n SELECT game_id,away_team_id,home_team_id,away_score,home_score FROM games\n), shot AS (\n SELECT game_id,team_id,COUNT(*) AS attempts,SUM(is_sog) AS sog,\n SUM(is_sog*is_slot) AS slot_sog,SUM(is_sog*is_inner_slot) AS inner_slot_sog,\n SUM(CASE WHEN position='D' THEN is_sog ELSE 0 END) AS defence_sog,\n SUM(CASE WHEN team_skaters=5 AND opponent_skaters=5 THEN is_sog ELSE 0 END) AS sog_5v5,\n SUM(CASE WHEN manpower='powerPlay' THEN is_sog ELSE 0 END) AS sog_pp,\n SUM(CASE WHEN manpower='shortHanded' THEN is_sog ELSE 0 END) AS sog_sh,\n SUM(xg_on_net) AS xg_on_net_feed, SUM(xg_all_shots) AS xg_all_shots_feed\n FROM shots WHERE is_shootout=0 GROUP BY game_id,team_id\n), metrics_wide AS (\n SELECT game_id,team_id,\n MAX(CASE WHEN metric_name='Expected Goals' THEN metric_value END) AS xg,\n MAX(CASE WHEN metric_name='Controlled Exits' THEN metric_value END) AS controlled_exits,\n MAX(CASE WHEN metric_name='Controlled Entries' THEN metric_value END) AS controlled_entries,\n MAX(CASE WHEN metric_name='OZ Face-Off Wins' THEN metric_value END) AS oz_faceoff_wins,\n MAX(CASE WHEN metric_name='OZ Face-Off Wins with Shot On Net After' THEN metric_value END) AS oz_faceoff_sog,\n MAX(CASE WHEN metric_name='OZ Face Off Win Shot On Net After Rate' THEN metric_value END) AS oz_faceoff_sog_rate,\n MAX(CASE WHEN metric_name='OZ Time of Possession' THEN metric_value END) AS oz_possession_seconds,\n MAX(CASE WHEN metric_name='Save Percentage' THEN metric_value END) AS save_percentage\n FROM team_metrics GROUP BY game_id,team_id\n), dq AS (\n SELECT game_id,SUM(CASE WHEN severity='error' THEN 1 ELSE 0 END) AS errors,\n SUM(CASE WHEN severity='warning' THEN 1 ELSE 0 END) AS warnings\n FROM quality GROUP BY game_id\n)\nSELECT s.game_id,g.competition_id,g.season,g.stage,g.game_date,s.team_id,t.team_name,s.opponent_id,o.team_name AS opponent,\n s.goals,s.goals_against,sh.attempts,sh.sog,against.sog AS sog_against,\n sh.slot_sog,against.slot_sog AS slot_sog_against,sh.inner_slot_sog,against.inner_slot_sog AS inner_slot_sog_against,\n sh.defence_sog,sh.sog_5v5,sh.sog_pp,sh.sog_sh,m.xg,ma.xg AS xg_against,\n sh.xg_on_net_feed,sh.xg_all_shots_feed,m.controlled_exits,m.controlled_entries,m.oz_faceoff_wins,m.oz_faceoff_sog,m.oz_faceoff_sog_rate,\n m.oz_possession_seconds,m.save_percentage,COALESCE(dq.errors,0) AS data_errors,COALESCE(dq.warnings,0) AS data_warnings,g.source_snapshot\nFROM sides s JOIN games g ON s.game_id=g.game_id\nLEFT JOIN teams t ON t.team_id=s.team_id LEFT JOIN teams o ON o.team_id=s.opponent_id\nLEFT JOIN shot sh ON sh.game_id=s.game_id AND sh.team_id=s.team_id\nLEFT JOIN shot against ON against.game_id=s.game_id AND against.team_id=s.opponent_id\nLEFT JOIN metrics_wide m ON m.game_id=s.game_id AND m.team_id=s.team_id\nLEFT JOIN metrics_wide ma ON ma.game_id=s.game_id AND ma.team_id=s.opponent_id\nLEFT JOIN dq ON dq.game_id=s.game_id;\n\nDROP VIEW IF EXISTS v_player_game;\nCREATE VIEW v_player_game AS\nWITH shooting AS (\n SELECT game_id,team_id,player_id,COUNT(*) AS attempts,SUM(is_sog) AS sog,SUM(is_goal) AS goals\n FROM shots WHERE is_shootout=0 GROUP BY game_id,team_id,player_id\n), ice AS (\n SELECT game_id,team_id,player_id,COUNT(*) AS shifts,SUM(duration_sec) AS toi_seconds,\n AVG(duration_sec) AS mean_shift_seconds,MAX(duration_sec) AS longest_shift_seconds\n FROM shifts GROUP BY game_id,team_id,player_id\n), bad_toi AS (\n SELECT DISTINCT game_id FROM quality WHERE check_name IN ('sparse_toi','empty_toi','unpaired_in','unpaired_out','open_shift','negative_shift','toi_identity')\n)\nSELECT r.game_id,g.game_date,g.season,r.team_id,t.team_name,r.player_id,p.player_name,r.jersey,r.position,\n CASE WHEN c.http_status=200 THEN COALESCE(sh.attempts,0) END AS attempts,\n CASE WHEN c.http_status=200 THEN COALESCE(sh.sog,0) END AS sog,\n CASE WHEN c.http_status=200 THEN COALESCE(sh.goals,0) END AS goals,\n CASE WHEN bad.game_id IS NULL THEN ice.shifts END AS shifts,\n CASE WHEN bad.game_id IS NULL THEN ice.toi_seconds END AS toi_seconds,\n CASE WHEN bad.game_id IS NULL THEN ice.mean_shift_seconds END AS mean_shift_seconds,\n CASE WHEN bad.game_id IS NULL THEN ice.longest_shift_seconds END AS longest_shift_seconds,\n CASE WHEN bad.game_id IS NULL THEN 'paired_intervals' ELSE 'review_toi' END AS toi_quality,g.source_snapshot\nFROM rosters r JOIN games g ON r.game_id=g.game_id\nLEFT JOIN players p ON p.player_id=r.player_id LEFT JOIN teams t ON t.team_id=r.team_id\nLEFT JOIN shooting sh ON sh.game_id=r.game_id AND sh.team_id=r.team_id AND sh.player_id=r.player_id\nLEFT JOIN ice ON ice.game_id=r.game_id AND ice.team_id=r.team_id AND ice.player_id=r.player_id\nLEFT JOIN bad_toi bad ON bad.game_id=r.game_id\nLEFT JOIN coverage c ON c.game_id=r.game_id AND c.endpoint='full';\n\nDROP VIEW IF EXISTS v_team_season;\nCREATE VIEW v_team_season AS\nSELECT competition_id,season,stage,team_id,team_name,COUNT(*) AS loaded_games,\n COUNT(sog) AS games_with_sog,COUNT(xg) AS games_with_xg,\n SUM(goals) AS goals,SUM(goals_against) AS goals_against,\n SUM(sog) AS sog,SUM(sog_against) AS sog_against,AVG(sog) AS sog_per_game,\n SUM(slot_sog) AS slot_sog,SUM(inner_slot_sog) AS inner_slot_sog,\n SUM(xg) AS xg,SUM(xg_against) AS xg_against,\n SUM(data_errors) AS data_errors,SUM(data_warnings) AS data_warnings\nFROM v_team_game GROUP BY competition_id,season,stage,team_id,team_name;\n"}
PACKAGE_ROOT = Path(tempfile.mkdtemp(prefix='almtuna_refresh10_'))
for relative, text in EMBEDDED_FILES.items():
    path = PACKAGE_ROOT/relative
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding='utf-8')
for module_name in list(sys.modules):
    if module_name == 'aj18_refresh10' or module_name.startswith('aj18_refresh10.'):
        del sys.modules[module_name]
sys.path.insert(0, str(PACKAGE_ROOT))
from aj18_refresh10 import api, model, ingestion as ingest
SOURCE_TABLES = list(model.SCHEMAS)
client = None

def api_client():
    global client
    if client is None:
        assert API_USERNAME.strip(), 'Fyll i det fungerande API_USERNAME i cell 1 och kör Run all.'
        cfg = {}
        candidate = api.Client('https://api.sportlogiq.com')
        try:
            cfg['SPORTLOGIQ_USERNAME'] = API_USERNAME
            cfg['SPORTLOGIQ_PASSWORD'] = dbutils.secrets.get(scope=SECRET_SCOPE, key='password')
            assert api.authenticate(candidate, cfg, allow_prompt=False), 'Inloggning misslyckades.'
            client = candidate
        finally:
            cfg.clear()
    return client

def close_client():
    global client
    if client is not None:
        client.auth.clear()
        client = None
    api.SECRETS.clear()

def build_status(schema):
    rows = spark.table(schema+'.build_status').collect()
    assert len(rows) == 1 and rows[0]['status'] == 'SUCCESS', 'Slutför eller återställ bygget i '+schema
    return rows[0]['build_id']

def table_version(name):
    if not spark.catalog.tableExists(name):
        return None
    return int(spark.sql(f'DESCRIBE HISTORY {name} LIMIT 1').first()['version'])

def save_run(status, **extra):
    record = {'run_id': RUN_ID, 'started_at': RUN_TIME, 'updated_at': datetime.now(timezone.utc).isoformat(), 'status': status, **extra}
    (RUN_DIR/'status.json').write_text(json.dumps(record, ensure_ascii=False, default=str, indent=2), encoding='utf-8')

PRE_BASE_ID = build_status(PROD_BASE)
PRE_INDIVIDUAL_ID = build_status(PROD_INDIVIDUAL)
production_snapshots = {str(r['game_key']):r['source_snapshot'] for r in spark.table(PROD_BASE+'.dim_game').collect()}
previous_individual_sources = spark.table(PROD_INDIVIDUAL+'.source_build_manifest').select('base_build_id').distinct().collect()
legacy_manifest = []
if len(previous_individual_sources)==1 and previous_individual_sources[0][0]==PRE_BASE_ID:
    legacy_manifest = [r.asDict() for r in spark.table(PROD_INDIVIDUAL+'.source_api_manifest').collect()]

# Versions of all tables that may be replaced: used for concurrency checks and recovery.
BASE_TABLES = ['dim_date','dim_team','dim_player','dim_competition_season','dim_game','dim_strength','dim_game_quality',
               'fact_team_game','fact_player_game','fact_shot','fact_shift','fact_event','fact_quality_issue','fact_api_coverage','source_manifest','build_status']
INDIVIDUAL_TABLES = ['fact_player_game','fact_player_metric','dim_metric','fact_metric_quality','source_api_manifest','source_build_manifest',
                     'dim_player','dim_game','dim_team','dim_date','dim_competition_season','build_status']
PASSING_TABLES = ['fact_game_team_pass_coverage','source_manifest','build_status']
GROUPS = [(WORK_BRONZE,PROD_BRONZE,['source_snapshots']),(WORK_SILVER,PROD_SILVER,SOURCE_TABLES),(WORK_BASE,PROD_BASE,BASE_TABLES),
          (WORK_PASSING,PROD_PASSING,PASSING_TABLES),(WORK_INDIVIDUAL,PROD_INDIVIDUAL,INDIVIDUAL_TABLES)]
BEFORE = {prod+'.'+name:table_version(prod+'.'+name) for work,prod,names in GROUPS for name in names}
assert all(BEFORE[PROD_SILVER+'.'+name] is not None for name in SOURCE_TABLES), 'Befintliga Silver-tabeller saknas.'
(RUN_DIR/'before_versions.json').write_text(json.dumps(BEFORE, indent=2), encoding='utf-8')
raw_index = ingest.current_raw(RAW, COMPETITION_ID, SEASON, STAGE, model.read_bundle)
assert raw_index, 'Inga råmatcher finns.'
assert set(production_snapshots).issubset(raw_index), 'Råarkivet saknar tidigare Gold-matcher. Återställ arkivet innan körning.'
# Existing result ZIP supplies definitions; missing definitions remain explicit, not invented.
files = sorted((VOLUME/'player_api_checks').glob('Almtuna_Player_API_224914_*.zip'))
assert files, 'Resultat-ZIP från notebook 04 saknas på volymen.'
seed_blob = files[-1].read_bytes()
with zipfile.ZipFile(io.BytesIO(seed_blob)) as z:
    seed = {n:json.loads(z.read(n)) for n in z.namelist() if n.endswith('.json')}
seed_topics = {str(d['response']['topic_id']):d['response'] for n,d in seed.items()
               if n.startswith('metric_') and n.endswith('_player.json') and d['request']['http_status']==200}
assert seed_topics, 'Pilotens ämneslista saknas.'
def walk(value):
    if isinstance(value, dict):
        yield value
        for v in value.values(): yield from walk(v)
    elif isinstance(value, list):
        for v in value: yield from walk(v)
definitions = {}
for n,d in seed.items():
    if n.startswith('definition_') and d['request']['http_status']==200:
        b=d['response']
        for m in walk(b.get('topic',{})):
            if 'key' in m and 'description' in m:
                definitions.setdefault(m['key'], []).append(dict(m,scope=b.get('scope'),topic_id=str(b.get('topic_id'))))
save_run('STARTED', previous_gold_build=PRE_BASE_ID, previous_individual_build=PRE_INDIVIDUAL_ID)
print('Kontrollerat råarkiv:', len(raw_index), 'matcher. Tidigare byggen är SUCCESS.')

## 3. Hämta nya eller korrigerade matcher
Oförändrade matcher återanvänds när API:ets processmarkör och endpoint-täckningen stämmer. Saknad processmarkör medför ny hämtning. Ett API-fel stoppar körningen före ändring av analystabellerna. Framgångsrika endpoints sparas för omkörning.

In [0]:
games = None
sync_result = []
TEAM_TOPICS = sorted(seed_topics, key=int) if not REFRESH_API else []
try:
    if REFRESH_API:
        active = api_client()
        games = ingest.list_games(active, COMPETITION_ID, SEASON, STAGE)
        unknown_forced = FORCED - set(games)
        assert not unknown_forced, 'Tvingade match-ID saknas i vald matchlista: '+str(sorted(unknown_forced))
        topic_payload = ingest.fetch_json(active, '/v1/hockey/metrictopics/team')
        TEAM_TOPICS = sorted(set(api.topic_ids(topic_payload)), key=int)
        assert TEAM_TOPICS, 'Tom ämneskatalog.'
        # Hela J18: arkiverade matcher (kontroll av ändringar) + nya matcher som Sportlogiq färdigprocessat. Ej processade/spelade utan markör hoppas över.
        candidate_games = {gid:g for gid,g in games.items() if gid in raw_index or g.get('last_metrics_full_process_time')}
        candidates = [gid for gid,g in candidate_games.items() if ingest.needs_fetch(g,raw_index.get(gid,(None,None))[1],TEAM_TOPICS,gid in FORCED)]
        print('Kandidater (hela J18):', len(candidates), 'av', len(games), 'J18-matcher — arkiverade:', len(raw_index))
        assert len(candidates)<=MAX_MATCHES_TO_FETCH, f'{len(candidates)} matcher kräver kontroll/hämtning. Höj MAX_MATCHES_TO_FETCH i cell 1 om detta är avsiktligt.'
        sync_result = ingest.sync_selected(active,games,raw_index,TEAM_TOPICS,candidates,RAW,CACHE,
                                          COMPETITION_ID,SEASON,STAGE,RUN_ID,FORCED,api.clean)
finally:
    close_client()
raw_paths = [raw_index[g][0] for g in sorted(raw_index)]
RAW_REVISIONS = {g:ingest.raw_revision(p) for g,(p,b) in raw_index.items()}
(RUN_DIR/'raw_manifest.json').write_text(json.dumps([{'game_id':g,'path':str(p),'sha256':RAW_REVISIONS[g],
    'snapshot':b['snapshot_id']} for g,(p,b) in sorted(raw_index.items())], indent=2), encoding='utf-8')
(RUN_DIR/'sync_result.json').write_text(json.dumps(sync_result, indent=2), encoding='utf-8')
print('Tillgängliga matcher för modellbygget:',len(raw_index))

## 4. Bygg arbets-Silver
SQLite skapas på lokal temporär disk. Endast JSONL läses in via volymen. Produktions-Silver och dashboarden ändras ännu inte.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType
local_model = Path(tempfile.mkdtemp(prefix='almtuna_model10_'))
try:
    tables = model.materialize([b for p,b in raw_index.values()], local_model)
    stage_files = RUN_DIR/'model'
    stage_files.mkdir(parents=True, exist_ok=True)
    for name in SOURCE_TABLES:
        shutil.copy2(local_model/(name+'.jsonl'),stage_files/(name+'.jsonl'))
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {WORK_BRONZE}')
    bronze_rows=[(gid,b['snapshot_id'],str(path),json.dumps(b,ensure_ascii=False)) for gid,(path,b) in sorted(raw_index.items())]
    spark.createDataFrame(bronze_rows,'game_id STRING, snapshot_id STRING, source_file STRING, payload_json STRING').write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable(WORK_BRONZE+'.source_snapshots')
    assert spark.table(WORK_BRONZE+'.source_snapshots').count()==len(raw_index)
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {WORK_SILVER}')
    types = {'TEXT':StringType(),'INTEGER':LongType(),'REAL':DoubleType()}
    for name, cols in model.SCHEMAS.items():
        schema = StructType([StructField(k,types[v],True) for k,v in cols.items()])
        df = spark.read.schema(schema).json(str(stage_files/(name+'.jsonl')))
        df.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable(WORK_SILVER+'.'+name)
        assert spark.table(WORK_SILVER+'.'+name).count()==len(tables[name]), 'Radantal avviker: '+name
    print({name:len(rows) for name,rows in tables.items()})
finally:
    shutil.rmtree(local_model, ignore_errors=True)


## 5. Bygg och kontrollera Gold-stjärnschemat
Samma 63 struktur- och källkontroller som i notebook 02 används. Kända måttfel maskeras enligt tidigare regler.

In [0]:
STAR_BUILD_STEPS = ["SOURCE_TABLES = ['games', 'teams', 'players', 'rosters', 'events', 'shots', 'toi_events', 'shifts', 'metrics', 'team_metrics', 'quality', 'coverage']\nsource_versions = {}\nfor name in SOURCE_TABLES:\n    full_name = f'{SILVER}.{name}'\n    version = int(spark.sql(f'DESCRIBE HISTORY {full_name} LIMIT 1').first()['version'])\n    source_versions[name] = version\n    spark.sql(f'CREATE OR REPLACE TEMP VIEW src_{name} AS SELECT * FROM {full_name} VERSION AS OF {version}')\nassert spark.sql('SELECT COUNT(*) FROM src_games').first()[0] > 0, 'Silver saknar matcher.'\nprint('Låsta Delta-versioner:', source_versions)\n", 'BASE_SQL = "-- Portable SQL: executed locally in SQLite and as Spark SQL views in Databricks.\\n\\n\\nCREATE OR REPLACE TEMP VIEW star_team_source AS\\nWITH sides AS (\\n SELECT game_id,home_team_id AS team_id,away_team_id AS opponent_id,home_score AS goals,away_score AS goals_against FROM src_games\\n UNION ALL\\n SELECT game_id,away_team_id,home_team_id,away_score,home_score FROM src_games\\n), shot AS (\\n SELECT game_id,team_id,COUNT(*) AS attempts,SUM(is_sog) AS sog,\\n SUM(is_sog*is_slot) AS slot_sog,SUM(is_sog*is_inner_slot) AS inner_slot_sog,\\n SUM(CASE WHEN position=\'D\' THEN is_sog ELSE 0 END) AS defence_sog,\\n SUM(CASE WHEN team_skaters=5 AND opponent_skaters=5 THEN is_sog ELSE 0 END) AS sog_5v5,\\n SUM(CASE WHEN manpower=\'powerPlay\' THEN is_sog ELSE 0 END) AS sog_pp,\\n SUM(CASE WHEN manpower=\'shortHanded\' THEN is_sog ELSE 0 END) AS sog_sh,\\n SUM(xg_on_net) AS xg_on_net_feed, SUM(xg_all_shots) AS xg_all_shots_feed\\n FROM src_shots WHERE is_shootout=0 GROUP BY game_id,team_id\\n), metrics_wide AS (\\n SELECT game_id,team_id,\\n MAX(CASE WHEN metric_name=\'Expected Goals\' THEN metric_value END) AS xg,\\n MAX(CASE WHEN metric_name=\'Controlled Exits\' THEN metric_value END) AS controlled_exits,\\n MAX(CASE WHEN metric_name=\'Controlled Entries\' THEN metric_value END) AS controlled_entries,\\n MAX(CASE WHEN metric_name=\'OZ Face-Off Wins\' THEN metric_value END) AS oz_faceoff_wins,\\n MAX(CASE WHEN metric_name=\'OZ Face-Off Wins with Shot On Net After\' THEN metric_value END) AS oz_faceoff_sog,\\n MAX(CASE WHEN metric_name=\'OZ Face Off Win Shot On Net After Rate\' THEN metric_value END) AS oz_faceoff_sog_rate,\\n MAX(CASE WHEN metric_name=\'OZ Time of Possession\' THEN metric_value END) AS oz_possession_seconds,\\n MAX(CASE WHEN metric_name=\'Save Percentage\' THEN metric_value END) AS save_percentage\\n FROM src_team_metrics GROUP BY game_id,team_id\\n), dq AS (\\n SELECT game_id,SUM(CASE WHEN severity=\'error\' THEN 1 ELSE 0 END) AS errors,\\n SUM(CASE WHEN severity=\'warning\' THEN 1 ELSE 0 END) AS warnings\\n FROM src_quality GROUP BY game_id\\n)\\nSELECT s.game_id,g.competition_id,g.season,g.stage,g.game_date,s.team_id,t.team_name,s.opponent_id,o.team_name AS opponent,\\n s.goals,s.goals_against,sh.attempts,sh.sog,against.sog AS sog_against,\\n sh.slot_sog,against.slot_sog AS slot_sog_against,sh.inner_slot_sog,against.inner_slot_sog AS inner_slot_sog_against,\\n sh.defence_sog,sh.sog_5v5,sh.sog_pp,sh.sog_sh,m.xg,ma.xg AS xg_against,\\n sh.xg_on_net_feed,sh.xg_all_shots_feed,m.controlled_exits,m.controlled_entries,m.oz_faceoff_wins,m.oz_faceoff_sog,m.oz_faceoff_sog_rate,\\n m.oz_possession_seconds,m.save_percentage,COALESCE(dq.errors,0) AS data_errors,COALESCE(dq.warnings,0) AS data_warnings,g.source_snapshot\\nFROM sides s JOIN src_games g ON s.game_id=g.game_id\\nLEFT JOIN src_teams t ON t.team_id=s.team_id LEFT JOIN src_teams o ON o.team_id=s.opponent_id\\nLEFT JOIN shot sh ON sh.game_id=s.game_id AND sh.team_id=s.team_id\\nLEFT JOIN shot against ON against.game_id=s.game_id AND against.team_id=s.opponent_id\\nLEFT JOIN metrics_wide m ON m.game_id=s.game_id AND m.team_id=s.team_id\\nLEFT JOIN metrics_wide ma ON ma.game_id=s.game_id AND ma.team_id=s.opponent_id\\nLEFT JOIN dq ON dq.game_id=s.game_id;\\n\\nCREATE OR REPLACE TEMP VIEW star_player_source AS\\nWITH shooting AS (\\n SELECT game_id,team_id,player_id,COUNT(*) AS attempts,SUM(is_sog) AS sog,SUM(is_goal) AS goals\\n FROM src_shots WHERE is_shootout=0 GROUP BY game_id,team_id,player_id\\n), ice AS (\\n SELECT game_id,team_id,player_id,COUNT(*) AS shifts,SUM(duration_sec) AS toi_seconds,\\n AVG(duration_sec) AS mean_shift_seconds,MAX(duration_sec) AS longest_shift_seconds\\n FROM src_shifts GROUP BY game_id,team_id,player_id\\n), bad_toi AS (\\n SELECT DISTINCT game_id FROM src_quality WHERE check_name IN (\'sparse_toi\',\'empty_toi\',\'unpaired_in\',\'unpaired_out\',\'open_shift\',\'negative_shift\',\'toi_identity\')\\n)\\nSELECT r.game_id,g.game_date,g.season,r.team_id,t.team_name,r.player_id,p.player_name,r.jersey,r.position,\\n CASE WHEN c.http_status=200 THEN COALESCE(sh.attempts,0) END AS attempts,\\n CASE WHEN c.http_status=200 THEN COALESCE(sh.sog,0) END AS sog,\\n CASE WHEN c.http_status=200 THEN COALESCE(sh.goals,0) END AS goals,\\n CASE WHEN bad.game_id IS NULL THEN ice.shifts END AS shifts,\\n CASE WHEN bad.game_id IS NULL THEN ice.toi_seconds END AS toi_seconds,\\n CASE WHEN bad.game_id IS NULL THEN ice.mean_shift_seconds END AS mean_shift_seconds,\\n CASE WHEN bad.game_id IS NULL THEN ice.longest_shift_seconds END AS longest_shift_seconds,\\n CASE WHEN bad.game_id IS NULL THEN \'paired_intervals\' ELSE \'review_toi\' END AS toi_quality,g.source_snapshot\\nFROM src_rosters r JOIN src_games g ON r.game_id=g.game_id\\nLEFT JOIN src_players p ON p.player_id=r.player_id LEFT JOIN src_teams t ON t.team_id=r.team_id\\nLEFT JOIN shooting sh ON sh.game_id=r.game_id AND sh.team_id=r.team_id AND sh.player_id=r.player_id\\nLEFT JOIN ice ON ice.game_id=r.game_id AND ice.team_id=r.team_id AND ice.player_id=r.player_id\\nLEFT JOIN bad_toi bad ON bad.game_id=r.game_id\\nLEFT JOIN src_coverage c ON c.game_id=r.game_id AND c.endpoint=\'full\';\\n\\nCREATE OR REPLACE TEMP VIEW star_season_source AS\\nSELECT competition_id,season,stage,team_id,team_name,COUNT(*) AS loaded_games,\\n COUNT(sog) AS games_with_sog,COUNT(xg) AS games_with_xg,\\n SUM(goals) AS goals,SUM(goals_against) AS goals_against,\\n SUM(sog) AS sog,SUM(sog_against) AS sog_against,AVG(sog) AS sog_per_game,\\n SUM(slot_sog) AS slot_sog,SUM(inner_slot_sog) AS inner_slot_sog,\\n SUM(xg) AS xg,SUM(xg_against) AS xg_against,\\n SUM(data_errors) AS data_errors,SUM(data_warnings) AS data_warnings\\nFROM star_team_source GROUP BY competition_id,season,stage,team_id,team_name;\\n\\n"\nfor statement in BASE_SQL.split(\';\'):\n    if statement.strip(): spark.sql(statement)', 'TABLE_QUERIES = {\'dim_date\': "SELECT DISTINCT CAST(replace(game_date,\'-\',\'\') AS INT) date_key,\\n      CAST(game_date AS DATE) calendar_date, year(CAST(game_date AS DATE)) calendar_year,\\n      month(CAST(game_date AS DATE)) calendar_month, day(CAST(game_date AS DATE)) day_of_month\\n      FROM src_games", \'dim_team\': "SELECT team_id team_key,team_id sportlogiq_team_id,team_name,team_shorthand FROM src_teams\\n      UNION ALL SELECT \'__unknown__\',NULL,\'Okänt lag\',NULL", \'dim_player\': "SELECT player_id player_key,player_id sportlogiq_player_id,player_name FROM src_players\\n      UNION ALL SELECT \'__unknown__\',NULL,\'Okänd spelare\'", \'dim_competition_season\': "SELECT DISTINCT concat(competition_id,\':\',season,\':\',stage) competition_season_key,\\n      competition_id,season,stage FROM src_games", \'dim_game\': \'SELECT game_id game_key,game_id sportlogiq_game_id,game_date,\\n      home_team_id,away_team_id,status,period_count,source_snapshot FROM src_games\', \'dim_strength\': "SELECT DISTINCT concat(COALESCE(manpower,\'?\'),\':\',COALESCE(CAST(team_skaters AS STRING),\'?\'),\':\',COALESCE(CAST(opponent_skaters AS STRING),\'?\')) strength_key,manpower,team_skaters,opponent_skaters FROM src_shots", \'dim_game_quality\': "WITH qc AS (\\n      SELECT game_id,\\n       SUM(CASE WHEN severity=\'error\' THEN 1 ELSE 0 END) error_count,\\n       SUM(CASE WHEN severity=\'warning\' THEN 1 ELSE 0 END) warning_count,\\n       MAX(CASE WHEN severity IN (\'error\',\'warning\') AND check_name IN\\n        (\'reconcile_sog\',\'feeds_sog\',\'reconcile_slot_sog\',\'reconcile_inner_slot_sog\',\'reconcile_goals\',\'feeds_goals\') THEN 1 ELSE 0 END) shot_issue,\\n       MAX(CASE WHEN severity IN (\'error\',\'warning\') AND check_name IN\\n        (\'sparse_toi\',\'empty_toi\',\'unpaired_in\',\'unpaired_out\',\'open_shift\',\'negative_shift\',\'toi_identity\') THEN 1 ELSE 0 END) toi_issue,\\n       MAX(CASE WHEN severity IN (\'error\',\'warning\') AND check_name NOT IN\\n        (\'reconcile_sog\',\'feeds_sog\',\'reconcile_slot_sog\',\'reconcile_inner_slot_sog\',\'reconcile_goals\',\'feeds_goals\',\\n         \'sparse_toi\',\'empty_toi\',\'unpaired_in\',\'unpaired_out\',\'open_shift\',\'negative_shift\',\'toi_identity\') THEN 1 ELSE 0 END) other_issue\\n      FROM src_quality GROUP BY game_id\\n     ), cov AS (\\n      SELECT game_id,MAX(CASE WHEN endpoint=\'full\' AND http_status=200 THEN 1 ELSE 0 END) full_ok,\\n      MAX(CASE WHEN http_status IS NULL OR http_status<>200 THEN 1 ELSE 0 END) api_issue\\n      FROM src_coverage GROUP BY game_id\\n     )\\n     SELECT g.game_id quality_key,g.game_id,\\n      COALESCE(q.error_count,0) error_count,COALESCE(q.warning_count,0) warning_count,\\n      CASE WHEN q.game_id IS NULL OR COALESCE(c.api_issue,1)=1 OR COALESCE(q.other_issue,0)=1 THEN 1 ELSE 0 END general_issue,\\n      CASE WHEN q.game_id IS NOT NULL AND COALESCE(c.api_issue,1)=0 AND COALESCE(q.other_issue,0)=0\\n        AND COALESCE(q.shot_issue,0)=0 AND COALESCE(c.full_ok,0)=1 THEN 1 ELSE 0 END shots_usable,\\n      CASE WHEN q.game_id IS NOT NULL AND COALESCE(c.api_issue,1)=0 AND COALESCE(q.other_issue,0)=0 THEN 1 ELSE 0 END xg_quality_pass,\\n      CASE WHEN q.game_id IS NOT NULL AND COALESCE(c.api_issue,1)=0 AND COALESCE(q.other_issue,0)=0\\n        AND COALESCE(q.toi_issue,0)=0 THEN 1 ELSE 0 END toi_usable,\\n      g.source_snapshot FROM src_games g LEFT JOIN qc q ON q.game_id=g.game_id LEFT JOIN cov c ON c.game_id=g.game_id", \'fact_team_game\': "SELECT concat(t.game_id,\':\',t.team_id) team_game_key,g.game_id game_key,CAST(replace(g.game_date,\'-\',\'\') AS INT) date_key,\\n      concat(g.competition_id,\':\',g.season,\':\',g.stage) competition_season_key,\\n      g.game_id quality_key,\\n      t.team_id team_key,t.opponent_id opponent_team_key,\\n      t.goals,t.goals_against,t.attempts,t.sog,t.sog_against,t.slot_sog,t.slot_sog_against,\\n      t.inner_slot_sog,t.inner_slot_sog_against,t.defence_sog,t.sog_5v5,t.sog_pp,t.sog_sh,\\n      t.xg,t.xg_against,t.xg_on_net_feed,t.xg_all_shots_feed,\\n      t.controlled_exits,t.controlled_entries,t.oz_faceoff_wins,t.oz_faceoff_sog,\\n      t.oz_faceoff_sog_rate,t.oz_possession_seconds,t.save_percentage,t.source_snapshot\\n      FROM star_team_source t JOIN src_games g ON g.game_id=t.game_id", \'fact_player_game\': "SELECT concat(p.game_id,\':\',p.team_id,\':\',p.player_id) player_game_key,g.game_id game_key,CAST(replace(g.game_date,\'-\',\'\') AS INT) date_key,\\n      concat(g.competition_id,\':\',g.season,\':\',g.stage) competition_season_key,\\n      g.game_id quality_key,\\n      p.team_id team_key,p.player_id player_key,p.jersey,p.position,\\n      p.attempts,p.sog,p.goals,p.shifts,p.toi_seconds,p.mean_shift_seconds,p.longest_shift_seconds,\\n      p.toi_quality,p.source_snapshot\\n      FROM star_player_source p JOIN src_games g ON g.game_id=p.game_id", \'fact_shot\': "SELECT s.shot_key,g.game_id game_key,CAST(replace(g.game_date,\'-\',\'\') AS INT) date_key,\\n      concat(g.competition_id,\':\',g.season,\':\',g.stage) competition_season_key,\\n      g.game_id quality_key,COALESCE(t.team_key,\'__unknown__\') team_key,\\n      COALESCE(p.player_key,\'__unknown__\') player_key,s.team_id source_team_id,s.player_id source_player_id,\\n      concat(COALESCE(s.manpower,\'?\'),\':\',COALESCE(CAST(s.team_skaters AS STRING),\'?\'),\':\',COALESCE(CAST(s.opponent_skaters AS STRING),\'?\')) strength_key,\\n      s.period,s.period_time,s.game_time,s.frame,s.position,s.zone,s.play_section,s.shot_type,s.outcome,s.x_adj,s.y_adj,s.is_sog,s.is_goal,s.is_slot,s.is_inner_slot,s.is_shootout,s.xg_on_net,s.xg_all_shots,s.source_snapshot\\n      FROM src_shots s JOIN src_games g ON g.game_id=s.game_id\\n      LEFT JOIN dim_team t ON t.team_key=s.team_id LEFT JOIN dim_player p ON p.player_key=s.player_id", \'fact_shift\': "SELECT s.shift_key,g.game_id game_key,CAST(replace(g.game_date,\'-\',\'\') AS INT) date_key,\\n      concat(g.competition_id,\':\',g.season,\':\',g.stage) competition_season_key,\\n      g.game_id quality_key,COALESCE(t.team_key,\'__unknown__\') team_key,\\n      COALESCE(p.player_key,\'__unknown__\') player_key,s.team_id source_team_id,s.player_id source_player_id,\\n      s.period,s.start_period_sec,s.end_period_sec,s.duration_sec,s.source_snapshot\\n      FROM src_shifts s JOIN src_games g ON g.game_id=s.game_id\\n      LEFT JOIN dim_team t ON t.team_key=s.team_id LEFT JOIN dim_player p ON p.player_key=s.player_id", \'fact_quality_issue\': "SELECT DISTINCT game_id game_key,game_id quality_key,check_name,severity,detail,source_snapshot FROM src_quality WHERE severity <> \'info\'", \'fact_api_coverage\': \'SELECT game_id game_key,endpoint,http_status,result,source_snapshot FROM src_coverage\'}\nVIEW_QUERIES = {\'v_team_game_safe\': \'SELECT f.*,t.team_name,o.team_name opponent,d.calendar_date,\\n q.error_count,q.warning_count,q.shots_usable,q.xg_quality_pass,q.toi_usable,\\n CASE WHEN q.shots_usable=1 THEN f.sog END sog_validated,\\n CASE WHEN q.shots_usable=1 THEN f.slot_sog END slot_sog_validated,\\n CASE WHEN q.shots_usable=1 THEN f.inner_slot_sog END inner_slot_sog_validated,\\n CASE WHEN q.xg_quality_pass=1 THEN f.xg END xg_validated\\n FROM fact_team_game f JOIN dim_game_quality q ON q.quality_key=f.quality_key\\n JOIN dim_team t ON t.team_key=f.team_key JOIN dim_team o ON o.team_key=f.opponent_team_key\\n JOIN dim_date d ON d.date_key=f.date_key\', \'v_player_game_safe\': \'SELECT f.*,p.player_name,t.team_name,q.shots_usable,q.toi_usable,\\n CASE WHEN q.shots_usable=1 THEN f.sog END sog_validated,\\n CASE WHEN q.toi_usable=1 THEN f.toi_seconds END toi_seconds_validated,\\n CASE WHEN q.toi_usable=1 THEN f.shifts END shifts_validated\\n FROM fact_player_game f JOIN dim_game_quality q ON q.quality_key=f.quality_key\\n JOIN dim_player p ON p.player_key=f.player_key JOIN dim_team t ON t.team_key=f.team_key\', \'v_team_season\': \'SELECT competition_season_key,team_key,team_name,COUNT(*) loaded_games,\\n COUNT(sog_validated) games_with_usable_sog,COUNT(xg_validated) games_with_usable_xg,\\n AVG(sog_validated) sog_per_game,AVG(slot_sog_validated) slot_sog_per_game,\\n AVG(xg_validated) xg_per_game,SUM(error_count) error_flags,SUM(warning_count) warning_flags\\n FROM v_team_game_safe GROUP BY competition_season_key,team_key,team_name\'}\nfor name, query in TABLE_QUERIES.items():\n    spark.sql(f\'CREATE OR REPLACE TEMP VIEW {name} AS {query}\')\nfor name, query in VIEW_QUERIES.items():\n    spark.sql(f\'CREATE OR REPLACE TEMP VIEW {name} AS {query}\')\nprint(\'Dimensioner och fakta är definierade.\')\n', "CHECKS = {'dim_date_unique': 'SELECT COUNT(*) FROM (SELECT date_key FROM dim_date GROUP BY date_key HAVING COUNT(*)>1) d', 'dim_date_not_null': 'SELECT COUNT(*) FROM dim_date WHERE date_key IS NULL', 'dim_team_unique': 'SELECT COUNT(*) FROM (SELECT team_key FROM dim_team GROUP BY team_key HAVING COUNT(*)>1) d', 'dim_team_not_null': 'SELECT COUNT(*) FROM dim_team WHERE team_key IS NULL', 'dim_player_unique': 'SELECT COUNT(*) FROM (SELECT player_key FROM dim_player GROUP BY player_key HAVING COUNT(*)>1) d', 'dim_player_not_null': 'SELECT COUNT(*) FROM dim_player WHERE player_key IS NULL', 'dim_game_unique': 'SELECT COUNT(*) FROM (SELECT game_key FROM dim_game GROUP BY game_key HAVING COUNT(*)>1) d', 'dim_game_not_null': 'SELECT COUNT(*) FROM dim_game WHERE game_key IS NULL', 'dim_competition_season_unique': 'SELECT COUNT(*) FROM (SELECT competition_season_key FROM dim_competition_season GROUP BY competition_season_key HAVING COUNT(*)>1) d', 'dim_competition_season_not_null': 'SELECT COUNT(*) FROM dim_competition_season WHERE competition_season_key IS NULL', 'dim_strength_unique': 'SELECT COUNT(*) FROM (SELECT strength_key FROM dim_strength GROUP BY strength_key HAVING COUNT(*)>1) d', 'dim_strength_not_null': 'SELECT COUNT(*) FROM dim_strength WHERE strength_key IS NULL', 'dim_game_quality_unique': 'SELECT COUNT(*) FROM (SELECT quality_key FROM dim_game_quality GROUP BY quality_key HAVING COUNT(*)>1) d', 'dim_game_quality_not_null': 'SELECT COUNT(*) FROM dim_game_quality WHERE quality_key IS NULL', 'fact_team_game_unique': 'SELECT COUNT(*) FROM (SELECT team_game_key FROM fact_team_game GROUP BY team_game_key HAVING COUNT(*)>1) d', 'fact_team_game_not_null': 'SELECT COUNT(*) FROM fact_team_game WHERE team_game_key IS NULL', 'fact_player_game_unique': 'SELECT COUNT(*) FROM (SELECT player_game_key FROM fact_player_game GROUP BY player_game_key HAVING COUNT(*)>1) d', 'fact_player_game_not_null': 'SELECT COUNT(*) FROM fact_player_game WHERE player_game_key IS NULL', 'fact_shot_unique': 'SELECT COUNT(*) FROM (SELECT shot_key FROM fact_shot GROUP BY shot_key HAVING COUNT(*)>1) d', 'fact_shot_not_null': 'SELECT COUNT(*) FROM fact_shot WHERE shot_key IS NULL', 'fact_shift_unique': 'SELECT COUNT(*) FROM (SELECT shift_key FROM fact_shift GROUP BY shift_key HAVING COUNT(*)>1) d', 'fact_shift_not_null': 'SELECT COUNT(*) FROM fact_shift WHERE shift_key IS NULL', 'fact_team_game_game_key': 'SELECT COUNT(*) FROM fact_team_game f LEFT JOIN dim_game d ON f.game_key=d.game_key WHERE d.game_key IS NULL', 'fact_team_game_date_key': 'SELECT COUNT(*) FROM fact_team_game f LEFT JOIN dim_date d ON f.date_key=d.date_key WHERE d.date_key IS NULL', 'fact_team_game_competition_season_key': 'SELECT COUNT(*) FROM fact_team_game f LEFT JOIN dim_competition_season d ON f.competition_season_key=d.competition_season_key WHERE d.competition_season_key IS NULL', 'fact_team_game_quality_key': 'SELECT COUNT(*) FROM fact_team_game f LEFT JOIN dim_game_quality d ON f.quality_key=d.quality_key WHERE d.quality_key IS NULL', 'fact_team_game_team_key': 'SELECT COUNT(*) FROM fact_team_game f LEFT JOIN dim_team d ON f.team_key=d.team_key WHERE d.team_key IS NULL', 'fact_team_game_opponent_team_key': 'SELECT COUNT(*) FROM fact_team_game f LEFT JOIN dim_team d ON f.opponent_team_key=d.team_key WHERE d.team_key IS NULL', 'fact_player_game_game_key': 'SELECT COUNT(*) FROM fact_player_game f LEFT JOIN dim_game d ON f.game_key=d.game_key WHERE d.game_key IS NULL', 'fact_player_game_date_key': 'SELECT COUNT(*) FROM fact_player_game f LEFT JOIN dim_date d ON f.date_key=d.date_key WHERE d.date_key IS NULL', 'fact_player_game_competition_season_key': 'SELECT COUNT(*) FROM fact_player_game f LEFT JOIN dim_competition_season d ON f.competition_season_key=d.competition_season_key WHERE d.competition_season_key IS NULL', 'fact_player_game_quality_key': 'SELECT COUNT(*) FROM fact_player_game f LEFT JOIN dim_game_quality d ON f.quality_key=d.quality_key WHERE d.quality_key IS NULL', 'fact_player_game_team_key': 'SELECT COUNT(*) FROM fact_player_game f LEFT JOIN dim_team d ON f.team_key=d.team_key WHERE d.team_key IS NULL', 'fact_player_game_player_key': 'SELECT COUNT(*) FROM fact_player_game f LEFT JOIN dim_player d ON f.player_key=d.player_key WHERE d.player_key IS NULL', 'fact_shot_game_key': 'SELECT COUNT(*) FROM fact_shot f LEFT JOIN dim_game d ON f.game_key=d.game_key WHERE d.game_key IS NULL', 'fact_shot_date_key': 'SELECT COUNT(*) FROM fact_shot f LEFT JOIN dim_date d ON f.date_key=d.date_key WHERE d.date_key IS NULL', 'fact_shot_competition_season_key': 'SELECT COUNT(*) FROM fact_shot f LEFT JOIN dim_competition_season d ON f.competition_season_key=d.competition_season_key WHERE d.competition_season_key IS NULL', 'fact_shot_quality_key': 'SELECT COUNT(*) FROM fact_shot f LEFT JOIN dim_game_quality d ON f.quality_key=d.quality_key WHERE d.quality_key IS NULL', 'fact_shot_team_key': 'SELECT COUNT(*) FROM fact_shot f LEFT JOIN dim_team d ON f.team_key=d.team_key WHERE d.team_key IS NULL', 'fact_shot_player_key': 'SELECT COUNT(*) FROM fact_shot f LEFT JOIN dim_player d ON f.player_key=d.player_key WHERE d.player_key IS NULL', 'fact_shot_strength_key': 'SELECT COUNT(*) FROM fact_shot f LEFT JOIN dim_strength d ON f.strength_key=d.strength_key WHERE d.strength_key IS NULL', 'fact_shift_game_key': 'SELECT COUNT(*) FROM fact_shift f LEFT JOIN dim_game d ON f.game_key=d.game_key WHERE d.game_key IS NULL', 'fact_shift_date_key': 'SELECT COUNT(*) FROM fact_shift f LEFT JOIN dim_date d ON f.date_key=d.date_key WHERE d.date_key IS NULL', 'fact_shift_competition_season_key': 'SELECT COUNT(*) FROM fact_shift f LEFT JOIN dim_competition_season d ON f.competition_season_key=d.competition_season_key WHERE d.competition_season_key IS NULL', 'fact_shift_quality_key': 'SELECT COUNT(*) FROM fact_shift f LEFT JOIN dim_game_quality d ON f.quality_key=d.quality_key WHERE d.quality_key IS NULL', 'fact_shift_team_key': 'SELECT COUNT(*) FROM fact_shift f LEFT JOIN dim_team d ON f.team_key=d.team_key WHERE d.team_key IS NULL', 'fact_shift_player_key': 'SELECT COUNT(*) FROM fact_shift f LEFT JOIN dim_player d ON f.player_key=d.player_key WHERE d.player_key IS NULL', 'two_teams_per_game': 'SELECT COUNT(*) FROM (SELECT game_key FROM fact_team_game GROUP BY game_key HAVING COUNT(*)<>2) d', 'shift_positive': 'SELECT COUNT(*) FROM fact_shift WHERE duration_sec<=0 OR duration_sec IS NULL', 'masked_sog': 'SELECT COUNT(*) FROM v_team_game_safe WHERE shots_usable=0 AND sog_validated IS NOT NULL', 'masked_toi': 'SELECT COUNT(*) FROM v_player_game_safe WHERE toi_usable=0 AND toi_seconds_validated IS NOT NULL', 'rosters_source_snapshot': 'SELECT COUNT(*) FROM src_rosters x LEFT JOIN src_games g ON g.game_id=x.game_id WHERE g.game_id IS NULL OR x.source_snapshot IS NULL OR x.source_snapshot<>g.source_snapshot', 'events_source_snapshot': 'SELECT COUNT(*) FROM src_events x LEFT JOIN src_games g ON g.game_id=x.game_id WHERE g.game_id IS NULL OR x.source_snapshot IS NULL OR x.source_snapshot<>g.source_snapshot', 'shots_source_snapshot': 'SELECT COUNT(*) FROM src_shots x LEFT JOIN src_games g ON g.game_id=x.game_id WHERE g.game_id IS NULL OR x.source_snapshot IS NULL OR x.source_snapshot<>g.source_snapshot', 'toi_events_source_snapshot': 'SELECT COUNT(*) FROM src_toi_events x LEFT JOIN src_games g ON g.game_id=x.game_id WHERE g.game_id IS NULL OR x.source_snapshot IS NULL OR x.source_snapshot<>g.source_snapshot', 'shifts_source_snapshot': 'SELECT COUNT(*) FROM src_shifts x LEFT JOIN src_games g ON g.game_id=x.game_id WHERE g.game_id IS NULL OR x.source_snapshot IS NULL OR x.source_snapshot<>g.source_snapshot', 'metrics_source_snapshot': 'SELECT COUNT(*) FROM src_metrics x LEFT JOIN src_games g ON g.game_id=x.game_id WHERE g.game_id IS NULL OR x.source_snapshot IS NULL OR x.source_snapshot<>g.source_snapshot', 'team_metrics_source_snapshot': 'SELECT COUNT(*) FROM src_team_metrics x LEFT JOIN src_games g ON g.game_id=x.game_id WHERE g.game_id IS NULL OR x.source_snapshot IS NULL OR x.source_snapshot<>g.source_snapshot', 'quality_source_snapshot': 'SELECT COUNT(*) FROM src_quality x LEFT JOIN src_games g ON g.game_id=x.game_id WHERE g.game_id IS NULL OR x.source_snapshot IS NULL OR x.source_snapshot<>g.source_snapshot', 'coverage_source_snapshot': 'SELECT COUNT(*) FROM src_coverage x LEFT JOIN src_games g ON g.game_id=x.game_id WHERE g.game_id IS NULL OR x.source_snapshot IS NULL OR x.source_snapshot<>g.source_snapshot', 'fact_shot_row_count': 'SELECT abs((SELECT COUNT(*) FROM fact_shot)-(SELECT COUNT(*) FROM src_shots))', 'fact_shift_row_count': 'SELECT abs((SELECT COUNT(*) FROM fact_shift)-(SELECT COUNT(*) FROM src_shifts))', 'fact_player_game_row_count': 'SELECT abs((SELECT COUNT(*) FROM fact_player_game)-(SELECT COUNT(*) FROM src_rosters))'}\ncheck_results = []\nfor name, query in CHECKS.items():\n    violations = int(spark.sql(query).first()[0])\n    check_results.append((name, violations))\nfailures = [(name, count) for name, count in check_results if count != 0]\nif failures:\n    display(spark.createDataFrame(failures, 'check_name STRING, violations LONG'))\n    raise RuntimeError('Strukturkontroller misslyckades. Inga Gold-tabeller har skrivits i denna körning.')\nprint(f'{len(check_results)} strukturkontroller passerade.')\ndisplay(spark.sql('SELECT * FROM dim_game_quality WHERE error_count>0 OR warning_count>0 ORDER BY game_id'))\n", 'for name, version in source_versions.items():\n    current = int(spark.sql(f\'DESCRIBE HISTORY {SILVER}.{name} LIMIT 1\').first()[\'version\'])\n    assert current == version, f\'Silver ändrades under bygget: {name}. Kör om notebooken efter avslutad inläsning.\'\nspark.sql(f\'CREATE SCHEMA IF NOT EXISTS {TARGET}\')\nspark.sql(f"CREATE OR REPLACE TABLE {TARGET}.build_status USING DELTA AS SELECT \'{BUILD_ID}\' build_id, \'{BUILD_TIME}\' build_time_utc, \'RUNNING\' status")\nspark.sql(f\'ALTER TABLE {TARGET}.build_status ALTER COLUMN build_id SET NOT NULL\')\nspark.sql(f\'ALTER TABLE {TARGET}.build_status ADD CONSTRAINT pk_build_status PRIMARY KEY (build_id)\')\nfor name in TABLE_QUERIES:\n    spark.sql(f"CREATE OR REPLACE TABLE {TARGET}.{name} USING DELTA AS SELECT *, \'{BUILD_ID}\' gold_build_id FROM {name}")\n\n# Primärnycklar för dimensions- och faktatabeller – eliminerar redundanta aggregeringar i unikhetskontrollerna.\nfor _pk in [(\'dim_date\',\'date_key\'),(\'dim_team\',\'team_key\'),(\'dim_player\',\'player_key\'),(\'dim_competition_season\',\'competition_season_key\'),(\'dim_game\',\'game_key\'),(\'dim_strength\',\'strength_key\'),(\'dim_game_quality\',\'quality_key\'),(\'fact_team_game\',\'team_game_key\'),(\'fact_player_game\',\'player_game_key\'),(\'fact_shot\',\'shot_key\'),(\'fact_shift\',\'shift_key\')]:\n    spark.sql(f\'ALTER TABLE {TARGET}.{_pk[0]} ALTER COLUMN {_pk[1]} SET NOT NULL\')\n    spark.sql(f\'ALTER TABLE {TARGET}.{_pk[0]} ADD CONSTRAINT pk_{_pk[0]} PRIMARY KEY ({_pk[1]})\')\n\n# Manifestet beskriver exakt vilka Silver-versioner som användes.\nmanifest = [(BUILD_ID, BUILD_TIME, f\'{SILVER}.{name}\', version) for name, version in source_versions.items()]\nspark.createDataFrame(manifest, \'build_id STRING, build_time_utc STRING, source_table STRING, delta_version LONG\').write.format(\'delta\').mode(\'overwrite\').saveAsTable(TARGET+\'.source_manifest\')\n\ndef qualify(query):\n    names = list(TABLE_QUERIES) + list(VIEW_QUERIES)\n    for name in sorted(names, key=len, reverse=True):\n        query = re.sub(r\'(?i)\\b(FROM|JOIN)\\s+\'+re.escape(name)+r\'\\b\', lambda m: m[1]+\' \'+TARGET+\'.\'+name, query)\n    return query\n\nfor name, query in VIEW_QUERIES.items():\n    spark.sql(f\'CREATE OR REPLACE VIEW {TARGET}.{name} AS {qualify(query)}\')\nfor name, query in CHECKS.items():\n    assert int(spark.sql(qualify(query)).first()[0]) == 0, name\nfor name in TABLE_QUERIES:\n    expected = spark.sql(f\'SELECT COUNT(*) FROM {name}\').first()[0]\n    actual = spark.sql(f\'SELECT COUNT(*) FROM {TARGET}.{name}\').first()[0]\n    assert actual == expected, name\nspark.sql(f"UPDATE {TARGET}.build_status SET status=\'SUCCESS\' WHERE build_id=\'{BUILD_ID}\'")\nprint(\'Gold-stjärnschemat är byggt:\', TARGET)\n']
star_env = {'spark':spark,'display':display,'re':re,'json':json,
            'SILVER':WORK_SILVER,'TARGET':WORK_BASE,'BUILD_ID':RUN_ID,'BUILD_TIME':RUN_TIME}
for step in STAR_BUILD_STEPS:
    exec(compile(step,'gold_star_builder','exec'),star_env)
# --- Promovera individual-mått till Gold ---
spark.sql(f"CREATE OR REPLACE TABLE {WORK_BASE}.fact_event USING DELTA AS "
    "SELECT e.event_key,g.game_id game_key,CAST(replace(g.game_date,'-','') AS INT) date_key,"
    "concat(g.competition_id,':',g.season,':',g.stage) competition_season_key,"
    "g.game_id quality_key,COALESCE(t.team_key,'__unknown__') team_key,"
    "COALESCE(p.player_key,'__unknown__') player_key,e.team_id source_team_id,e.player_id source_player_id,"
    "e.event_name,e.event_type,e.outcome,e.zone,e.period,e.period_time,e.game_time,"
    "e.manpower,e.team_skaters,e.opponent_skaters,e.possession_id,"
    "e.is_shot_attempt,e.is_sog,e.is_goal,e.is_shootout,e.source_snapshot,"
    f"'{RUN_ID}' gold_build_id "
    "FROM src_events e JOIN src_games g ON g.game_id=e.game_id "
    f"LEFT JOIN {WORK_BASE}.dim_team t ON t.team_key=e.team_id "
    f"LEFT JOIN {WORK_BASE}.dim_player p ON p.player_key=e.player_id")
spark.sql(f'ALTER TABLE {WORK_BASE}.fact_event ALTER COLUMN event_key SET NOT NULL')
spark.sql(f'ALTER TABLE {WORK_BASE}.fact_event ADD CONSTRAINT pk_fact_event PRIMARY KEY (event_key)')

spark.sql(f"""
CREATE OR REPLACE VIEW {WORK_BASE}.v_player_event_game AS
WITH evt AS (
  SELECT game_key, player_key, team_key,
    SUM(CASE WHEN event_name='faceoff' AND outcome='successful' THEN 1 ELSE 0 END) AS faceoff_wins,
    SUM(CASE WHEN event_name='faceoff' AND outcome='failed' THEN 1 ELSE 0 END) AS faceoff_losses,
    SUM(CASE WHEN event_name='faceoff' AND outcome='successful' AND zone='oz' THEN 1 ELSE 0 END) AS fo_wins_oz,
    SUM(CASE WHEN event_name='faceoff' AND outcome='successful' AND zone='nz' THEN 1 ELSE 0 END) AS fo_wins_nz,
    SUM(CASE WHEN event_name='faceoff' AND outcome='successful' AND zone='dz' THEN 1 ELSE 0 END) AS fo_wins_dz,
    SUM(CASE WHEN event_name='faceoff' AND outcome='failed' AND zone='oz' THEN 1 ELSE 0 END) AS fo_losses_oz,
    SUM(CASE WHEN event_name='faceoff' AND outcome='failed' AND zone='nz' THEN 1 ELSE 0 END) AS fo_losses_nz,
    SUM(CASE WHEN event_name='faceoff' AND outcome='failed' AND zone='dz' THEN 1 ELSE 0 END) AS fo_losses_dz,
    SUM(CASE WHEN event_name='penalty' THEN 1 ELSE 0 END) AS penalties_taken,
    SUM(CASE WHEN event_name='penalty' AND zone='oz' THEN 1 ELSE 0 END) AS penalties_oz,
    SUM(CASE WHEN event_name='penalty' AND zone='dz' THEN 1 ELSE 0 END) AS penalties_dz,
    ARRAY_JOIN(ARRAY_DISTINCT(COLLECT_LIST(CASE WHEN event_name='penalty' THEN event_type END)), ', ') AS penalty_types,
    SUM(CASE WHEN event_name='lpr' AND zone='oz' THEN 1 ELSE 0 END) AS oz_recoveries,
    SUM(CASE WHEN event_name='lpr' AND zone='dz' THEN 1 ELSE 0 END) AS dz_recoveries,
    SUM(CASE WHEN event_name='lpr' AND zone='nz' THEN 1 ELSE 0 END) AS nz_recoveries,
    SUM(CASE WHEN event_name='lpr' AND event_type='faceoffcontested' THEN 1 ELSE 0 END) AS contested_recoveries,
    SUM(CASE WHEN event_name='lpr' AND zone='oz' AND event_type NOT IN ('faceoff','faceoffcontested','rebound') THEN 1 ELSE 0 END) AS oz_forecheck_wins,
    SUM(CASE WHEN event_name='lpr' AND zone='dz' AND event_type NOT IN ('faceoff','faceoffcontested','rebound') THEN 1 ELSE 0 END) AS dz_def_recoveries,
    SUM(CASE WHEN event_name='controlledexit' THEN 1 ELSE 0 END) AS controlled_exits_evt,
    SUM(CASE WHEN event_name='zoneexit' AND outcome='successful' THEN 1 ELSE 0 END) AS zone_exits_ok,
    SUM(CASE WHEN event_name='zoneexit' AND outcome='failed' THEN 1 ELSE 0 END) AS zone_exits_failed,
    SUM(CASE WHEN event_name='breakout' AND outcome='successful' THEN 1 ELSE 0 END) AS breakouts_ok,
    SUM(CASE WHEN event_name='breakout' AND outcome='failed' THEN 1 ELSE 0 END) AS breakouts_failed,
    SUM(CASE WHEN event_name='zoneexit' AND outcome='successful' AND manpower='shortHanded' THEN 1 ELSE 0 END) AS pk_exits_ok,
    SUM(CASE WHEN event_name='zoneexit' AND outcome='failed' AND manpower='shortHanded' THEN 1 ELSE 0 END) AS pk_exits_failed,
    SUM(CASE WHEN event_name='controlledentry' THEN 1 ELSE 0 END) AS controlled_entries_evt,
    SUM(CASE WHEN event_name='carry' THEN 1 ELSE 0 END) AS carries,
    SUM(CASE WHEN event_name='endtoendrush' THEN 1 ELSE 0 END) AS end_to_end_rushes,
    SUM(CASE WHEN event_name='controlledentry' AND manpower='powerPlay' THEN 1 ELSE 0 END) AS pp_entries,
    SUM(CASE WHEN event_name='scoringchance' AND team_skaters=5 AND opponent_skaters=5 THEN 1 ELSE 0 END) AS scoring_chances_5v5,
    SUM(CASE WHEN event_name='scoringchance' AND manpower='powerPlay' THEN 1 ELSE 0 END) AS scoring_chances_pp,
    SUM(CASE WHEN event_name='scoringchance' THEN 1 ELSE 0 END) AS scoring_chances_all,
    SUM(CASE WHEN event_name='lpr' THEN 1 ELSE 0 END) AS puck_wins,
    SUM(CASE WHEN event_name='lpr' AND (event_type LIKE '%shotonnet%' OR event_type LIKE '%slotshot%') THEN 1 ELSE 0 END) AS puck_wins_with_shot
  FROM {WORK_BASE}.fact_event
  WHERE player_key IS NOT NULL AND player_key != '__unknown__'
  GROUP BY game_key, player_key, team_key
),
shot AS (
  SELECT game_key, player_key,
    SUM(CASE WHEN strength_key='evenStrength:5:5' THEN xg_on_net ELSE 0 END) AS xg_5v5,
    SUM(CASE WHEN strength_key LIKE 'powerPlay%' THEN xg_on_net ELSE 0 END) AS xg_pp,
    SUM(CASE WHEN strength_key='evenStrength:5:5' THEN is_sog ELSE 0 END) AS sog_5v5,
    SUM(CASE WHEN strength_key LIKE 'powerPlay%' THEN is_sog ELSE 0 END) AS sog_pp
  FROM {WORK_BASE}.fact_shot
  WHERE is_shootout=0
  GROUP BY game_key, player_key
)
SELECT d.calendar_date, e.game_key, e.player_key, p.player_name, pg.position,
       e.team_key, tg.opponent_team_key,
       e.faceoff_wins, e.faceoff_losses,
       CASE WHEN e.faceoff_wins+e.faceoff_losses>0
            THEN ROUND(e.faceoff_wins*100.0/(e.faceoff_wins+e.faceoff_losses),1)
            ELSE NULL END AS faceoff_pct,
       e.fo_wins_oz, e.fo_wins_nz, e.fo_wins_dz,
       e.fo_losses_oz, e.fo_losses_nz, e.fo_losses_dz,
       e.penalties_taken, e.penalties_oz, e.penalties_dz, e.penalty_types,
       e.oz_recoveries, e.dz_recoveries, e.nz_recoveries, e.contested_recoveries,
       e.oz_forecheck_wins, e.dz_def_recoveries,
       e.controlled_exits_evt, e.zone_exits_ok, e.zone_exits_failed,
       e.breakouts_ok, e.breakouts_failed, e.pk_exits_ok, e.pk_exits_failed,
       e.controlled_entries_evt, e.carries, e.end_to_end_rushes, e.pp_entries,
       e.scoring_chances_5v5, e.scoring_chances_pp, e.scoring_chances_all,
       COALESCE(s.xg_5v5,0) AS xg_5v5, COALESCE(s.xg_pp,0) AS xg_pp,
       COALESCE(s.sog_5v5,0) AS sog_5v5, COALESCE(s.sog_pp,0) AS sog_pp,
       e.puck_wins, e.puck_wins_with_shot
FROM evt e
JOIN {WORK_BASE}.dim_player p ON p.player_key=e.player_key
JOIN {WORK_BASE}.fact_player_game pg ON pg.game_key=e.game_key AND pg.player_key=e.player_key
JOIN {WORK_BASE}.fact_team_game tg ON tg.game_key=e.game_key AND tg.team_key=e.team_key
JOIN {WORK_BASE}.dim_date d ON d.date_key=pg.date_key
LEFT JOIN shot s ON s.game_key=e.game_key AND s.player_key=e.player_key
""")

spark.sql(f"""
CREATE OR REPLACE VIEW {WORK_BASE}.v_player_onice_xg_game AS
WITH onice AS (
  SELECT s.player_key, s.game_key, s.team_key,
    sh.xg_on_net, sh.is_sog, sh.is_slot,
    CASE WHEN sh.team_key=s.team_key THEN 1 ELSE 0 END AS is_for
  FROM {WORK_BASE}.fact_shift s
  JOIN {WORK_BASE}.fact_shot sh
    ON sh.game_key=s.game_key AND sh.period=s.period
    AND sh.period_time >= s.start_period_sec AND sh.period_time <= s.end_period_sec
  WHERE sh.strength_key='evenStrength:5:5' AND sh.is_shootout=0
)
SELECT d.calendar_date, o.game_key, o.player_key, p.player_name, pg.position,
       pg.team_key, tg.opponent_team_key,
       SUM(CASE WHEN o.is_for=1 THEN o.xg_on_net ELSE 0 END) AS xg_for_5v5,
       SUM(CASE WHEN o.is_for=0 THEN o.xg_on_net ELSE 0 END) AS xg_against_5v5,
       SUM(CASE WHEN o.is_for=1 THEN o.xg_on_net ELSE 0 END) - SUM(CASE WHEN o.is_for=0 THEN o.xg_on_net ELSE 0 END) AS xg_diff_5v5,
       SUM(CASE WHEN o.is_for=1 THEN 1 ELSE 0 END) AS shots_for_5v5,
       SUM(CASE WHEN o.is_for=0 THEN 1 ELSE 0 END) AS shots_against_5v5,
       SUM(CASE WHEN o.is_for=1 THEN o.is_sog ELSE 0 END) AS sog_for_5v5,
       SUM(CASE WHEN o.is_for=0 THEN o.is_sog ELSE 0 END) AS sog_against_5v5,
       SUM(CASE WHEN o.is_for=1 AND o.is_slot=1 THEN 1 ELSE 0 END) AS slot_shots_for_5v5,
       SUM(CASE WHEN o.is_for=0 AND o.is_slot=1 THEN 1 ELSE 0 END) AS slot_shots_against_5v5
FROM onice o
JOIN {WORK_BASE}.dim_player p ON p.player_key=o.player_key
JOIN {WORK_BASE}.fact_player_game pg ON pg.game_key=o.game_key AND pg.player_key=o.player_key
JOIN {WORK_BASE}.fact_team_game tg ON tg.game_key=o.game_key AND tg.team_key=o.team_key
JOIN {WORK_BASE}.dim_date d ON d.date_key=pg.date_key
GROUP BY d.calendar_date, o.game_key, o.player_key, p.player_name, pg.position, pg.team_key, tg.opponent_team_key
""")

spark.sql(f"""
CREATE OR REPLACE VIEW {WORK_BASE}.v_player_event_season AS
WITH event_season AS (
  SELECT player_key, player_name,
    ARRAY_JOIN(COLLECT_SET(position), '/') AS positions,
    COUNT(*) AS games,
    SUM(faceoff_wins) AS faceoff_wins, SUM(faceoff_losses) AS faceoff_losses,
    SUM(fo_wins_oz) AS fo_wins_oz, SUM(fo_wins_nz) AS fo_wins_nz, SUM(fo_wins_dz) AS fo_wins_dz,
    SUM(fo_losses_oz) AS fo_losses_oz, SUM(fo_losses_nz) AS fo_losses_nz, SUM(fo_losses_dz) AS fo_losses_dz,
    SUM(penalties_taken) AS penalties_taken, SUM(penalties_oz) AS penalties_oz, SUM(penalties_dz) AS penalties_dz,
    ARRAY_JOIN(ARRAY_DISTINCT(FLATTEN(COLLECT_LIST(CASE WHEN penalty_types IS NOT NULL THEN SPLIT(penalty_types, ', ') ELSE ARRAY() END))), ', ') AS penalty_types_all,
    SUM(oz_recoveries) AS oz_recoveries, SUM(dz_recoveries) AS dz_recoveries, SUM(nz_recoveries) AS nz_recoveries,
    SUM(contested_recoveries) AS contested_recoveries,
    SUM(oz_forecheck_wins) AS oz_forecheck_wins, SUM(dz_def_recoveries) AS dz_def_recoveries,
    SUM(controlled_exits_evt) AS controlled_exits_evt, SUM(zone_exits_ok) AS zone_exits_ok, SUM(zone_exits_failed) AS zone_exits_failed,
    SUM(breakouts_ok) AS breakouts_ok, SUM(breakouts_failed) AS breakouts_failed,
    SUM(pk_exits_ok) AS pk_exits_ok, SUM(pk_exits_failed) AS pk_exits_failed,
    SUM(controlled_entries_evt) AS controlled_entries_evt, SUM(carries) AS carries,
    SUM(end_to_end_rushes) AS end_to_end_rushes, SUM(pp_entries) AS pp_entries,
    SUM(scoring_chances_5v5) AS scoring_chances_5v5, SUM(scoring_chances_pp) AS scoring_chances_pp, SUM(scoring_chances_all) AS scoring_chances_all,
    SUM(xg_5v5) AS xg_5v5, SUM(xg_pp) AS xg_pp, SUM(sog_5v5) AS sog_5v5, SUM(sog_pp) AS sog_pp,
    SUM(puck_wins) AS puck_wins, SUM(puck_wins_with_shot) AS puck_wins_with_shot
  FROM {WORK_BASE}.v_player_event_game
  GROUP BY player_key, player_name
),
onice_season AS (
  SELECT player_key,
    SUM(xg_for_5v5) AS xg_for_5v5, SUM(xg_against_5v5) AS xg_against_5v5,
    SUM(shots_for_5v5) AS shots_for_5v5, SUM(shots_against_5v5) AS shots_against_5v5,
    SUM(sog_for_5v5) AS sog_for_5v5, SUM(sog_against_5v5) AS sog_against_5v5,
    SUM(slot_shots_for_5v5) AS slot_shots_for_5v5, SUM(slot_shots_against_5v5) AS slot_shots_against_5v5
  FROM {WORK_BASE}.v_player_onice_xg_game
  GROUP BY player_key
)
SELECT e.*,
       CASE WHEN e.faceoff_wins+e.faceoff_losses>0
            THEN ROUND(e.faceoff_wins*100.0/(e.faceoff_wins+e.faceoff_losses),1)
            ELSE NULL END AS faceoff_pct,
       o.xg_for_5v5, o.xg_against_5v5,
       CASE WHEN o.xg_for_5v5 IS NOT NULL THEN o.xg_for_5v5-o.xg_against_5v5 END AS xg_diff_5v5,
       o.shots_for_5v5, o.shots_against_5v5,
       o.sog_for_5v5, o.sog_against_5v5,
       o.slot_shots_for_5v5, o.slot_shots_against_5v5
FROM event_season e
LEFT JOIN onice_season o ON o.player_key=e.player_key
""")

print('Event-mått promoverade till Gold:', WORK_BASE)

assert build_status(WORK_BASE)==RUN_ID
print('Gold klart i arbetsschemat.')

## 6. Bedöm passningsdata för alla lag och matcher
Den gamla piloten behövs inte för denna kontroll. Resultatet byggs direkt från samma Silver-version som Gold. Täckningen har en rad per match och lag.

In [0]:
PASSING_SQL = "WITH passes AS (\n SELECT game_id,team_id,COUNT(*) registered_passes,\n SUM(CASE WHEN outcome IN ('successful','failed') THEN 1 ELSE 0 END) classified_passes,\n SUM(CASE WHEN player_id IS NULL OR player_id='' THEN 1 ELSE 0 END) passes_without_player\n FROM cov_events WHERE event_name='pass' GROUP BY game_id,team_id\n)\nSELECT g.game_key,g.team_key,g.date_key,g.competition_season_key,\n COALESCE(p.registered_passes,0) registered_passes,\n COALESCE(p.classified_passes,0) classified_passes,\n COALESCE(p.passes_without_player,0) passes_without_player,\n CASE WHEN p.registered_passes>0 THEN CAST(p.classified_passes AS DOUBLE)/p.registered_passes END outcome_coverage,\n CASE WHEN p.registered_passes IS NULL THEN 'NO_RECORDED_EVENTS'\n      WHEN p.classified_passes=0 THEN 'OUTCOMES_UNCLASSIFIED'\n      WHEN p.classified_passes<p.registered_passes THEN 'PARTIAL_OUTCOMES'\n      ELSE 'RECORDED_OUTCOMES_CLASSIFIED' END passing_status,\n 'UNKNOWN' full_tracking_completeness\nFROM cov_team_games g LEFT JOIN passes p ON g.game_key=p.game_id AND g.team_key=p.team_id"
spark.table(WORK_BASE+'.fact_team_game').createOrReplaceTempView('cov_team_games')
event_version = int(spark.sql(f"SELECT delta_version FROM {WORK_BASE}.source_manifest WHERE source_table='{WORK_SILVER}.events'").first()[0])
spark.read.format('delta').option('versionAsOf',str(event_version)).table(WORK_SILVER+'.events').createOrReplaceTempView('cov_events')
assert spark.sql('SELECT COUNT(*) FROM (SELECT event_key FROM cov_events GROUP BY event_key HAVING COUNT(*)>1)').first()[0]==0
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {WORK_PASSING}')
spark.sql(PASSING_SQL).withColumn('gold_build_id',F.lit(RUN_ID)).write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable(WORK_PASSING+'.fact_game_team_pass_coverage')
passing = spark.table(WORK_PASSING+'.fact_game_team_pass_coverage')
assert passing.count()==spark.table(WORK_BASE+'.fact_team_game').count()
assert passing.groupBy('game_key','team_key').count().filter('count<>1').count()==0
assert passing.filter('classified_passes>registered_passes OR classified_passes<0').count()==0
spark.createDataFrame([(RUN_ID,WORK_SILVER+'.events',event_version,RUN_ID)],
    'base_build_id STRING, source_table STRING, delta_version LONG, gold_build_id STRING').write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable(WORK_PASSING+'.source_manifest')
spark.sql(f"CREATE OR REPLACE TABLE {WORK_PASSING}.build_status USING DELTA AS SELECT '{RUN_ID}' build_id,'{RUN_TIME}' build_time_utc,'SUCCESS' status")
display(passing.groupBy('passing_status').agg(F.count('*').alias('team_games'),F.sum('registered_passes').alias('registered_passes'),F.sum('classified_passes').alias('classified_passes')))

## 7. Individuella mått för Almtuna och topplagen
Matchantalet har inget fyramatchstak. API-svar kopplas till råmatchens SHA-256. Vid ändrad lagkälla hämtas även spelarvärdena om. Tidigare verifierade cachefiler återanvänds bara om Gold-snapshot och sparad filhash stämmer. Saknade definitioner markeras uttryckligen.

In [0]:
import math,json
from collections import defaultdict
SKATmER={'sog':'Total Successful Shots Taken','goals':'Total Regular Goals','slot_sog':'Total Successful Shot From Slot Attempts','inner_slot_sog':'Total Successful Shots Taken From Inner Slot','xg':'Expected Goals','controlled_exits':'Controlled Exits','controlled_entries':'Controlled Entries'}
GOALIE={'shots_against':'Total Shots On Net Against','saves':'Total Saves','goals_against':'Total Regular Goals Against','slot_shots_against':'Total Shots From Slot On Net Against','slot_saves':'Total Saves From Slot Shots'}
def unpack(game_id,roster_payload,topics):
    roster={};values={};origins=defaultdict(set)
    for tid,b in roster_payload.items():
        if not isinstance(b,dict):continue
        for p in b.get('players',[]):
            pid=str(p['id'])
            if pid in roster:raise ValueError('Duplicerat roster-ID '+pid)
            roster[pid]={'game_key':game_id,'player_key':pid,'team_key':str(tid),'position':p.get('position') or 'UNKNOWN','player_name':(p.get('first_name','')+' '+p.get('last_name','')).strip()}
    if not roster:raise ValueError('Tom roster '+game_id)
    for topic,b in topics.items():
        if str(b.get('id'))!=game_id or str(b.get('topic_id'))!=topic:raise ValueError('Fel match/topic')
        if not isinstance(b.get('groupings'),list):raise ValueError('Fel svarsformat')
        seen=set()
        for g in b['groupings']:
            dims={d['key']:str(d['value']) for d in g['breakdown']}
            if set(dims)!={'player'}:raise ValueError('Fel grain')
            pid=dims['player']
            if pid not in roster or pid in seen:raise ValueError('Okänt/duplicerat spelar-ID')
            seen.add(pid)
            for m in g.get('metrics',[]):
                v=m['value'];key=(pid,m['key'])
                if v is not None:
                    if isinstance(v,bool) or not isinstance(v,(float,int)) or not math.isfinite(v):raise ValueError('Ej numeriskt mått')
                    v=float(v)
                if key in values and values[key]!=v:raise ValueError('Motstridiga mått '+str(key))
                values[key]=v;origins[key].add(topic)
    metrics=[dict(game_key=game_id,player_key=pid,team_key=roster[pid]['team_key'],metric_key=k,value=v,source_topics=','.join(sorted(origins[(pid,k)],key=int))) for (pid,k),v in sorted(values.items())]
    rows=[]
    for pid,r in sorted(roster.items()):
        row=dict(r)
        row['metrics_returned']=int(any(p==pid for p,k in values))
        for col,key in {**SKATER,**GOALIE}.items():row[col]=values.get((pid,key))
        rows.append(row)
    return rows,metrics


In [0]:
BASE = WORK_BASE
COVERAGE = WORK_PASSING
TARGET = WORK_INDIVIDUAL
BASE_ID = build_status(BASE)
COVERAGE_ID = build_status(COVERAGE)
BUILD_ID = RUN_ID
BUILD_TIME = RUN_TIME
status = build_status
# Samla små tabeller till en stabil kopia och kontrollera bygg-ID.
def frozen(name,expected):
    df=spark.table(name);rows=df.collect()
    assert rows and {r['gold_build_id'] for r in rows}=={expected}, 'Blandade källbyggen: '+name
    return [r.asDict() for r in rows]
teams=frozen(BASE+'.v_team_game_safe',BASE_ID)
game_ids=sorted({r['game_key'] for r in teams if r['team_key'] in FETCH_TEAM_IDS})
assert game_ids, 'Inga matcher för valda lag finns i Gold.'
teams=[r for r in teams if r['game_key'] in game_ids]
base_players=[r for r in frozen(BASE+'.v_player_game_safe',BASE_ID) if r['game_key'] in game_ids]
pass_coverage=[r for r in frozen(COVERAGE+'.fact_game_team_pass_coverage',COVERAGE_ID) if r['game_key'] in game_ids]
assert status(BASE)==BASE_ID and status(COVERAGE)==COVERAGE_ID
team_lookup={(r['game_key'],r['team_key']):r for r in teams}
player_lookup={(r['game_key'],r['player_key'],r['team_key']):r for r in base_players}
pass_lookup={(r['game_key'],r['team_key']):r for r in pass_coverage}
print('Matcher:',game_ids)

TOPICS = sorted(set(seed_topics)|set(TEAM_TOPICS),key=int)
source_records = []
all_roster = []
all_metrics = []
try:
    for gid in game_ids:
        bundle = raw_index[gid][1]
        revision = RAW_REVISIONS[gid]
        root = CACHE/'individual'/gid/revision
        legacy = ingest.legacy_entries(gid,bundle['snapshot_id'],production_snapshots,legacy_manifest)
        def player_payload(topic=None):
            key = 'roster' if topic is None else topic
            filename = 'roster.json' if topic is None else 'topic_'+topic+'.json'
            path = root/filename
            endpoint = '/v1/hockey/games/'+gid+('/roster' if topic is None else '/metrics/'+topic)
            params = None if topic is None else {'breakdown':['player']}
            if not path.exists() and key in legacy and gid not in FORCED:
                old = legacy[key]
                entry = {'revision':revision,'endpoint':endpoint,'params':params,
                         'retrieved_at':old['retrieved_at'],'response':old['response'],
                         'response_sha256':ingest.digest(old['response'])}
                path.parent.mkdir(parents=True,exist_ok=True)
                path.write_text(json.dumps(entry,ensure_ascii=False,allow_nan=False),encoding='utf-8')
            # A lazy proxy avoids logging in when every required response is cached.
            class LazyClient:
                def call(self,*args,**kwargs):return api_client().call(*args,**kwargs)
            entry = ingest.cached_fetch(LazyClient(),endpoint,params,path,revision,api.clean)
            raw = path.read_bytes()
            source_records.append({'game_key':gid,'topic':key,'path':str(path),
                                   'sha256':hashlib.sha256(raw).hexdigest(),'retrieved_at':entry['retrieved_at']})
            return entry['response']
        roster_rows,metric_rows = unpack(gid,player_payload(),{topic:player_payload(topic) for topic in TOPICS})
        assert {r['team_key'] for r in roster_rows}=={r['team_key'] for r in teams if r['game_key']==gid}, 'Fel lag i spelarunderlaget.'
        assert len({r['team_key'] for r in roster_rows})==2
        all_roster.extend(roster_rows)
        all_metrics.extend(metric_rows)
        print('Spelarunderlag klart:',gid,'—',len(roster_rows),'trupprader',flush=True)
finally:
    close_client()
assert all_metrics, 'Inga individuella mått.'


## 8. Kontrollera individer och bygg analysvyer
Måttsummeringar kontrolleras mot lagtotalerna. Ett spärrat mått blir NULL; råvärdena behålls i den långa måttfaktan. Detta garanterar inte att varje enskild händelse attribuerats till rätt spelare.

In [0]:
checks=[];metric_ok={}
for gt,base in team_lookup.items():
    rr=[r for r in all_roster if (r['game_key'],r['team_key'])==gt and r['metrics_returned']==1]
    for col in ['sog','goals','slot_sog','inner_slot_sog','xg']:
        vals=[r[col] for r in rr]
        usable=base['xg_quality_pass']==1 if col=='xg' else base['shots_usable']==1
        ok=bool(vals) and all(v is not None and v>=0 for v in vals) and base[col] is not None and abs(sum(vals)-float(base[col]))<1e-6 and usable
        metric_ok[gt+(col,)]=ok
        checks.append({'game_key':gt[0],'team_key':gt[1],'metric':col,'status':'RECONCILED' if ok else 'BLOCKED','api_total':sum(v for v in vals if v is not None) if vals else None,'gold_total':float(base[col]) if base[col] is not None else None})
for r in all_roster:
    gt=(r['game_key'],r['team_key']);base=team_lookup[gt]
    bp=player_lookup.get((r['game_key'],r['player_key'],r['team_key']))
    toi=bp.get('toi_seconds_validated') if bp else None
    r['toi_seconds']=float(toi) if toi is not None and toi>0 else None
    r['participation_status']='POSITIVE_VALIDATED_TOI' if r['toi_seconds'] is not None else 'PARTICIPATION_UNCONFIRMED'
    r['role']='GOALIE' if r['position']=='G' else ('SKATER' if r['position'] in ('D','C','LW','RW','F') else 'UNKNOWN')
    r['date_key']=base['date_key'];r['competition_season_key']=base['competition_season_key'];r['opponent_team_key']=base['opponent_team_key']
    r['calendar_date']=str(base['calendar_date']);r['team_name']=base['team_name']
    cov=pass_lookup.get(gt,{})
    r['passing_status']=cov.get('passing_status','NOT_ASSESSED')
    r['successful_passes']=None;r['failed_passes']=None
    for col in ['sog','goals','slot_sog','inner_slot_sog','xg']:
        valid=metric_ok[gt+(col,)] and r['metrics_returned']==1 and r[col] is not None
        r[col+'_status']='RECONCILED' if valid else 'BLOCKED_OR_MISSING'
        if not valid:r[col]=None
    a,s,g=r['shots_against'],r['saves'],r['goals_against']
    sa,ss=r['slot_shots_against'],r['slot_saves']
    goalie_ok=r['role']=='GOALIE' and base['shots_usable']==1 and all(v is not None and v>=0 for v in [a,s,g,sa,ss]) and s+g==a and ss<=sa<=a and ss<=s
    r['goalie_status']='ARITHMETIC_AND_GAME_QUALITY_PASS' if goalie_ok else 'BLOCKED_OR_MISSING'
    for col in GOALIE:
        if not goalie_ok:r[col]=None
# Participation denominators count positive quality-checked TOI, not roster rows.
print('Måttkontroller:',{s:sum(r['status']==s for r in checks) for s in ['RECONCILED','BLOCKED']})


In [0]:
assert status(BASE)==BASE_ID and status(COVERAGE)==COVERAGE_ID
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {TARGET}')
spark.sql(f"CREATE OR REPLACE TABLE {TARGET}.build_status USING DELTA AS SELECT '{BUILD_ID}' build_id,'{BUILD_TIME}' build_time_utc,'RUNNING' status")
spark.sql(f'ALTER TABLE {TARGET}.build_status ALTER COLUMN build_id SET NOT NULL')
spark.sql(f'ALTER TABLE {TARGET}.build_status ADD CONSTRAINT pk_build_status PRIMARY KEY (build_id)')
def write(name,rows,schema):
    spark.createDataFrame(rows,schema).withColumn('gold_build_id',F.lit(BUILD_ID)).write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable(TARGET+'.'+name)
common='game_key STRING, player_key STRING, team_key STRING'
strings=['position','player_name','role','participation_status','competition_season_key','opponent_team_key','calendar_date','team_name','passing_status','goalie_status']+[k+'_status' for k in ['sog','goals','slot_sog','inner_slot_sog','xg']]
doubles=list(SKATER)+list(GOALIE)+['toi_seconds','successful_passes','failed_passes']
schema=common+', date_key INT, metrics_returned INT, '+', '.join(k+' STRING' for k in strings)+', '+', '.join(k+' DOUBLE' for k in doubles)
write('fact_player_game',all_roster,schema)
write('fact_player_metric',all_metrics,common+', metric_key STRING, value DOUBLE, source_topics STRING')
keys=sorted({r['metric_key'] for r in all_metrics})
write('dim_metric',[{'metric_key':k,'definitions_json':json.dumps(definitions.get(k,[]),ensure_ascii=False),'definition_status':'AVAILABLE' if k in definitions else 'MISSING','aggregation_rule':'NOT_APPROVED_USE_CURATED_VIEWS'} for k in keys],'metric_key STRING, definitions_json STRING, definition_status STRING, aggregation_rule STRING')
write('fact_metric_quality',checks,'game_key STRING, team_key STRING, metric STRING, status STRING, api_total DOUBLE, gold_total DOUBLE')
write('source_api_manifest',source_records,'game_key STRING, topic STRING, path STRING, sha256 STRING, retrieved_at STRING')
write('source_build_manifest',[{'base_build_id':BASE_ID,'coverage_build_id':COVERAGE_ID,'definition_zip':str(files[-1]),'definition_sha256':hashlib.sha256(seed_blob).hexdigest()}],'base_build_id STRING, coverage_build_id STRING, definition_zip STRING, definition_sha256 STRING')
# Shared dimensions copied from the consistent base build. Player names come from source roster.
players={}
for r in all_roster:players[r['player_key']]=r['player_name']
write('dim_player',[{'player_key':k,'player_name':v} for k,v in players.items()],'player_key STRING, player_name STRING')
for dim,key,values in [('dim_game','game_key',game_ids),('dim_team','team_key',list({r['team_key'] for r in all_roster})),('dim_date','date_key',list({r['date_key'] for r in all_roster})),('dim_competition_season','competition_season_key',list({r['competition_season_key'] for r in all_roster}))]:
    df=spark.table(BASE+'.'+dim).filter(F.col(key).isin(values))
    assert {r[0] for r in df.select('gold_build_id').distinct().collect()}=={BASE_ID}
    df.drop('gold_build_id').withColumn('gold_build_id',F.lit(BUILD_ID)).write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable(TARGET+'.'+dim)
for role,view in [('SKATER','v_skater_game'),('GOALIE','v_goalie_game')]:
    spark.sql(f"""CREATE OR REPLACE VIEW {TARGET}.{view} AS SELECT f.* FROM {TARGET}.fact_player_game f
    JOIN {TARGET}.build_status b ON f.gold_build_id=b.build_id AND b.status='SUCCESS' WHERE f.role='{role}'""")
spark.sql(f"""CREATE OR REPLACE VIEW {TARGET}.v_skater_season AS
 SELECT competition_season_key,team_key,player_key,MAX(player_name) player_name,
 COUNT(*) roster_games,
 SUM(CASE WHEN participation_status='POSITIVE_VALIDATED_TOI' THEN 1 ELSE 0 END) games_with_positive_validated_toi,
 COUNT(CASE WHEN toi_seconds>0 THEN sog END) games_with_usable_sog,
 AVG(CASE WHEN toi_seconds>0 THEN sog END) sog_per_usable_game,
 COUNT(CASE WHEN toi_seconds>0 THEN xg END) games_with_usable_xg,
 SUM(CASE WHEN toi_seconds>0 THEN xg END) xg,
 AVG(CASE WHEN toi_seconds>0 THEN xg END) xg_per_usable_game,
 SUM(toi_seconds)/60.0 validated_toi_minutes
 FROM {TARGET}.v_skater_game GROUP BY competition_season_key,team_key,player_key""")
spark.sql(f"""CREATE OR REPLACE VIEW {TARGET}.v_goalie_season AS
 SELECT competition_season_key,team_key,player_key,MAX(player_name) player_name,COUNT(*) roster_games,
 SUM(CASE WHEN toi_seconds>0 THEN 1 ELSE 0 END) games_with_positive_validated_toi,
 COUNT(CASE WHEN toi_seconds>0 THEN shots_against END) games_with_usable_goalie_stats,
 SUM(CASE WHEN toi_seconds>0 THEN shots_against END) shots_against,
 SUM(CASE WHEN toi_seconds>0 THEN saves END) saves,
 SUM(CASE WHEN toi_seconds>0 THEN goals_against END) goals_against,
 CASE WHEN SUM(CASE WHEN toi_seconds>0 THEN shots_against END)>0 THEN
 SUM(CASE WHEN toi_seconds>0 THEN saves END)/SUM(CASE WHEN toi_seconds>0 THEN shots_against END) END save_pct
 FROM {TARGET}.v_goalie_game GROUP BY competition_season_key,team_key,player_key""")


In [0]:
assert spark.table(TARGET+'.fact_player_game').count()==len(all_roster)
for table,pk in [('fact_player_game',['game_key','player_key','team_key']),('fact_player_metric',['game_key','player_key','metric_key'])]:
    df=spark.table(TARGET+'.'+table)
    assert df.groupBy(*pk).count().filter('count>1').count()==0,'Dubbletter: '+table
    for key,dim in [('game_key','dim_game'),('player_key','dim_player'),('team_key','dim_team')]:
        assert df.join(spark.table(TARGET+'.'+dim).select(key),key,'left_anti').count()==0,'Saknad dimensionsnyckel'
assert status(BASE)==BASE_ID and status(COVERAGE)==COVERAGE_ID
spark.sql(f"UPDATE {TARGET}.build_status SET status='SUCCESS' WHERE build_id='{BUILD_ID}'")

print('Individbygget klarade nycklar, relationer och byggkontroller.')

## 9. Slutkontroll före tabellbyte
Tidigare matchhistorik måste finnas kvar och samma produktionsversioner ska fortfarande gälla. Om API:ets processmarkör ändrats under hämtningen stoppas körningen före tabellbytet.

In [0]:
assert build_status(WORK_BASE)==RUN_ID and build_status(WORK_INDIVIDUAL)==RUN_ID
new_games = {r[0] for r in spark.table(WORK_BASE+'.dim_game').select('game_key').collect()}
assert set(production_snapshots).issubset(new_games), 'Tidigare matcher skulle försvinna.'
old_player_games = {r[0] for r in spark.table(PROD_INDIVIDUAL+'.dim_game').select('game_key').collect()}
assert old_player_games.issubset(set(game_ids)), 'Tidigare individuella matcher skulle försvinna.'
assert spark.table(WORK_INDIVIDUAL+'.fact_player_game').filter(F.col('team_key')==TEAM_ID).count()>0
for gid,(path,bundle) in raw_index.items():
    assert ingest.raw_revision(path)==RAW_REVISIONS[gid], 'Råfil ändrades under körning: '+gid
if games is not None:
    try:
        end_games=ingest.list_games(api_client(),COMPETITION_ID,SEASON,STAGE)
        for gid in set(games)&new_games:
            assert gid in end_games, 'Match försvann ur API-listan under körning: '+gid
            assert games[gid].get('last_metrics_full_process_time')==end_games[gid].get('last_metrics_full_process_time'), 'API-data ändrades under körning för '+gid+'. Kör om.'
    finally:
        close_client()
for name,version in BEFORE.items():
    assert table_version(name)==version, 'Samtidigt ändrad produktion: '+name+'. Kör om när övriga byggen avslutats.'
for schema in (PROD_BASE,PROD_INDIVIDUAL):
    for view in (['v_team_game_safe','v_player_game_safe','v_team_season'] if schema==PROD_BASE else ['v_skater_game','v_skater_season','v_goalie_game','v_goalie_season']):
        assert spark.catalog.tableExists(schema+'.'+view), 'Befintlig dashboardvy saknas: '+schema+'.'+view
# Require unchanged schemas on dashboard-facing existing facts/dimensions.
for work,prod,names in GROUPS:
    for name in names:
        if BEFORE[prod+'.'+name] is not None and name not in ('source_manifest','source_build_manifest','build_status'):
            old_columns={f.name:f.dataType.simpleString() for f in spark.table(prod+'.'+name).schema.fields}
            new_columns={f.name:f.dataType.simpleString() for f in spark.table(work+'.'+name).schema.fields}
            assert old_columns==new_columns, 'Schema har ändrats; granska innan byte: '+prod+'.'+name
save_run('VALIDATED',games=len(new_games),individual_games=len(game_ids))
print('Kontroller klara. Nästa cell uppdaterar analystabellerna.')

## 10. Uppdatera befintliga analystabeller
Tabellernas namn behålls så att befintliga dashboardfrågor fortsätter fungera. Byggstatus sätts till SUCCESS sist. Produktionsmanifesten skrivs med de faktiska nya Silver-versionerna. Dashboarddefinitionen ändras inte.

In [0]:
changed = []
production_versions = {}
def track_change(name):
    if name not in changed:
        changed.append(name)
        (RUN_DIR/'changed_tables.json').write_text(json.dumps(changed,indent=2),encoding='utf-8')
def copy_table(source,target):
    track_change(target)
    spark.sql(f'CREATE OR REPLACE TABLE {target} USING DELTA AS SELECT * FROM {source}')
    assert spark.table(source).count()==spark.table(target).count(), 'Radantal avviker vid byte: '+target

def restore_previous():
    failures=[]
    # Keep existing analysis builds visibly incomplete if recovery itself fails.
    for name in changed:
        if name.endswith('.build_status') and BEFORE[name] is not None:
            try:
                spark.sql(f"UPDATE {name} SET status='RUNNING'")
            except Exception as exc:
                failures.append({'table':name,'error_type':type(exc).__name__})
    # Restore data first and status tables last. Keep the recovery log after failures.
    order=[n for n in reversed(changed) if not n.endswith('.build_status')]+[n for n in changed if n.endswith('.build_status')]
    for name in order:
        if name.endswith('.build_status') and failures:
            continue  # Never report the old SUCCESS after a failed data recovery.
        version=BEFORE[name]
        try:
            if version is None:
                spark.sql(f'DROP TABLE IF EXISTS {name}')
            else:
                spark.sql(f'RESTORE TABLE {name} TO VERSION AS OF {version}')
        except Exception as exc:
            failures.append({'table':name,'error_type':type(exc).__name__})
    (RUN_DIR/'recovery.json').write_text(json.dumps({'failures':failures},indent=2),encoding='utf-8')
    return failures

try:
    save_run('PROMOTING')
    for work,prod,names in GROUPS:
        spark.sql(f'CREATE SCHEMA IF NOT EXISTS {prod}')
    # Mark the two existing analysis builds before replacing any data.
    for prod in (PROD_BASE,PROD_INDIVIDUAL):
        track_change(prod+'.build_status')
        spark.sql(f"CREATE OR REPLACE TABLE {prod}.build_status USING DELTA AS SELECT '{RUN_ID}' build_id,'{RUN_TIME}' build_time_utc,'RUNNING' status")
    copy_table(WORK_BRONZE+'.source_snapshots',PROD_BRONZE+'.source_snapshots')
    for name in SOURCE_TABLES:
        copy_table(WORK_SILVER+'.'+name,PROD_SILVER+'.'+name)
        production_versions[name]=table_version(PROD_SILVER+'.'+name)
    for name in BASE_TABLES:
        if name not in ('build_status','source_manifest'):
            copy_table(WORK_BASE+'.'+name,PROD_BASE+'.'+name)
    spark.sql(f'ALTER TABLE {PROD_BASE}.fact_event ALTER COLUMN event_key SET NOT NULL')
    spark.sql(f'ALTER TABLE {PROD_BASE}.fact_event ADD CONSTRAINT pk_fact_event PRIMARY KEY (event_key)')
    track_change(PROD_BASE+'.source_manifest')
    manifest=[(RUN_ID,RUN_TIME,PROD_SILVER+'.'+name,version) for name,version in production_versions.items()]
    spark.createDataFrame(manifest,'build_id STRING, build_time_utc STRING, source_table STRING, delta_version LONG').write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable(PROD_BASE+'.source_manifest')
    copy_table(WORK_PASSING+'.fact_game_team_pass_coverage',PROD_PASSING+'.fact_game_team_pass_coverage')
    track_change(PROD_PASSING+'.source_manifest')
    spark.createDataFrame([(RUN_ID,PROD_SILVER+'.events',production_versions['events'],RUN_ID)],
        'base_build_id STRING, source_table STRING, delta_version LONG, gold_build_id STRING').write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable(PROD_PASSING+'.source_manifest')
    for name in INDIVIDUAL_TABLES:
        if name!='build_status':copy_table(WORK_INDIVIDUAL+'.'+name,PROD_INDIVIDUAL+'.'+name)
    # Verify copied build identities and the actual dashboard view queries before SUCCESS.
    for prod,names in [(PROD_BASE,BASE_TABLES),(PROD_INDIVIDUAL,INDIVIDUAL_TABLES)]:
        for name in names:
            if name not in ('build_status','source_manifest'):
                ids={r[0] for r in spark.table(prod+'.'+name).select('gold_build_id').distinct().collect()}
                assert ids.issubset({RUN_ID}), 'Blandade bygg-ID: '+prod+'.'+name
    for name,version in production_versions.items():
        assert table_version(PROD_SILVER+'.'+name)==version, 'Silver ändrades samtidigt.'
    for work,prod in [(WORK_BASE,PROD_BASE),(WORK_PASSING,PROD_PASSING),(WORK_INDIVIDUAL,PROD_INDIVIDUAL)]:
        copy_table(work+'.build_status',prod+'.build_status')
    # Promote individual event metrics to Gold star schema.
    spark.sql(f"""
    CREATE OR REPLACE VIEW {PROD_BASE}.v_player_event_game AS
    WITH evt AS (
      SELECT game_key, player_key, team_key,
        SUM(CASE WHEN event_name='faceoff' AND outcome='successful' THEN 1 ELSE 0 END) AS faceoff_wins,
        SUM(CASE WHEN event_name='faceoff' AND outcome='failed' THEN 1 ELSE 0 END) AS faceoff_losses,
        SUM(CASE WHEN event_name='faceoff' AND outcome='successful' AND zone='oz' THEN 1 ELSE 0 END) AS fo_wins_oz,
        SUM(CASE WHEN event_name='faceoff' AND outcome='successful' AND zone='nz' THEN 1 ELSE 0 END) AS fo_wins_nz,
        SUM(CASE WHEN event_name='faceoff' AND outcome='successful' AND zone='dz' THEN 1 ELSE 0 END) AS fo_wins_dz,
        SUM(CASE WHEN event_name='faceoff' AND outcome='failed' AND zone='oz' THEN 1 ELSE 0 END) AS fo_losses_oz,
        SUM(CASE WHEN event_name='faceoff' AND outcome='failed' AND zone='nz' THEN 1 ELSE 0 END) AS fo_losses_nz,
        SUM(CASE WHEN event_name='faceoff' AND outcome='failed' AND zone='dz' THEN 1 ELSE 0 END) AS fo_losses_dz,
        SUM(CASE WHEN event_name='penalty' THEN 1 ELSE 0 END) AS penalties_taken,
        SUM(CASE WHEN event_name='penalty' AND zone='oz' THEN 1 ELSE 0 END) AS penalties_oz,
        SUM(CASE WHEN event_name='penalty' AND zone='dz' THEN 1 ELSE 0 END) AS penalties_dz,
        ARRAY_JOIN(ARRAY_DISTINCT(COLLECT_LIST(CASE WHEN event_name='penalty' THEN event_type END)), ', ') AS penalty_types,
        SUM(CASE WHEN event_name='lpr' AND zone='oz' THEN 1 ELSE 0 END) AS oz_recoveries,
        SUM(CASE WHEN event_name='lpr' AND zone='dz' THEN 1 ELSE 0 END) AS dz_recoveries,
        SUM(CASE WHEN event_name='lpr' AND zone='nz' THEN 1 ELSE 0 END) AS nz_recoveries,
        SUM(CASE WHEN event_name='lpr' AND event_type='faceoffcontested' THEN 1 ELSE 0 END) AS contested_recoveries,
        SUM(CASE WHEN event_name='lpr' AND zone='oz' AND event_type NOT IN ('faceoff','faceoffcontested','rebound') THEN 1 ELSE 0 END) AS oz_forecheck_wins,
        SUM(CASE WHEN event_name='lpr' AND zone='dz' AND event_type NOT IN ('faceoff','faceoffcontested','rebound') THEN 1 ELSE 0 END) AS dz_def_recoveries,
        SUM(CASE WHEN event_name='controlledexit' THEN 1 ELSE 0 END) AS controlled_exits_evt,
        SUM(CASE WHEN event_name='zoneexit' AND outcome='successful' THEN 1 ELSE 0 END) AS zone_exits_ok,
        SUM(CASE WHEN event_name='zoneexit' AND outcome='failed' THEN 1 ELSE 0 END) AS zone_exits_failed,
        SUM(CASE WHEN event_name='breakout' AND outcome='successful' THEN 1 ELSE 0 END) AS breakouts_ok,
        SUM(CASE WHEN event_name='breakout' AND outcome='failed' THEN 1 ELSE 0 END) AS breakouts_failed,
        SUM(CASE WHEN event_name='zoneexit' AND outcome='successful' AND manpower='shortHanded' THEN 1 ELSE 0 END) AS pk_exits_ok,
        SUM(CASE WHEN event_name='zoneexit' AND outcome='failed' AND manpower='shortHanded' THEN 1 ELSE 0 END) AS pk_exits_failed,
        SUM(CASE WHEN event_name='controlledentry' THEN 1 ELSE 0 END) AS controlled_entries_evt,
        SUM(CASE WHEN event_name='carry' THEN 1 ELSE 0 END) AS carries,
        SUM(CASE WHEN event_name='endtoendrush' THEN 1 ELSE 0 END) AS end_to_end_rushes,
        SUM(CASE WHEN event_name='controlledentry' AND manpower='powerPlay' THEN 1 ELSE 0 END) AS pp_entries,
        SUM(CASE WHEN event_name='scoringchance' AND team_skaters=5 AND opponent_skaters=5 THEN 1 ELSE 0 END) AS scoring_chances_5v5,
        SUM(CASE WHEN event_name='scoringchance' AND manpower='powerPlay' THEN 1 ELSE 0 END) AS scoring_chances_pp,
        SUM(CASE WHEN event_name='scoringchance' THEN 1 ELSE 0 END) AS scoring_chances_all,
        SUM(CASE WHEN event_name='lpr' THEN 1 ELSE 0 END) AS puck_wins,
        SUM(CASE WHEN event_name='lpr' AND (event_type LIKE '%shotonnet%' OR event_type LIKE '%slotshot%') THEN 1 ELSE 0 END) AS puck_wins_with_shot
      FROM {PROD_BASE}.fact_event
      WHERE player_key IS NOT NULL AND player_key != '__unknown__'
      GROUP BY game_key, player_key, team_key
    ),
    shot AS (
      SELECT game_key, player_key,
        SUM(CASE WHEN strength_key='evenStrength:5:5' THEN xg_on_net ELSE 0 END) AS xg_5v5,
        SUM(CASE WHEN strength_key LIKE 'powerPlay%' THEN xg_on_net ELSE 0 END) AS xg_pp,
        SUM(CASE WHEN strength_key='evenStrength:5:5' THEN is_sog ELSE 0 END) AS sog_5v5,
        SUM(CASE WHEN strength_key LIKE 'powerPlay%' THEN is_sog ELSE 0 END) AS sog_pp
      FROM {PROD_BASE}.fact_shot
      WHERE is_shootout=0
      GROUP BY game_key, player_key
    )
    SELECT d.calendar_date, e.game_key, e.player_key, p.player_name, pg.position,
           e.team_key, tg.opponent_team_key,
           e.faceoff_wins, e.faceoff_losses,
           CASE WHEN e.faceoff_wins+e.faceoff_losses>0
                THEN ROUND(e.faceoff_wins*100.0/(e.faceoff_wins+e.faceoff_losses),1)
                ELSE NULL END AS faceoff_pct,
           e.fo_wins_oz, e.fo_wins_nz, e.fo_wins_dz,
           e.fo_losses_oz, e.fo_losses_nz, e.fo_losses_dz,
           e.penalties_taken, e.penalties_oz, e.penalties_dz, e.penalty_types,
           e.oz_recoveries, e.dz_recoveries, e.nz_recoveries, e.contested_recoveries,
           e.oz_forecheck_wins, e.dz_def_recoveries,
           e.controlled_exits_evt, e.zone_exits_ok, e.zone_exits_failed,
           e.breakouts_ok, e.breakouts_failed, e.pk_exits_ok, e.pk_exits_failed,
           e.controlled_entries_evt, e.carries, e.end_to_end_rushes, e.pp_entries,
           e.scoring_chances_5v5, e.scoring_chances_pp, e.scoring_chances_all,
           COALESCE(s.xg_5v5,0) AS xg_5v5, COALESCE(s.xg_pp,0) AS xg_pp,
           COALESCE(s.sog_5v5,0) AS sog_5v5, COALESCE(s.sog_pp,0) AS sog_pp,
           e.puck_wins, e.puck_wins_with_shot
    FROM evt e
    JOIN {PROD_BASE}.dim_player p ON p.player_key=e.player_key
    JOIN {PROD_BASE}.fact_player_game pg ON pg.game_key=e.game_key AND pg.player_key=e.player_key
    JOIN {PROD_BASE}.fact_team_game tg ON tg.game_key=e.game_key AND tg.team_key=e.team_key
    JOIN {PROD_BASE}.dim_date d ON d.date_key=pg.date_key
    LEFT JOIN shot s ON s.game_key=e.game_key AND s.player_key=e.player_key
    """)
    spark.sql(f"""
    CREATE OR REPLACE VIEW {PROD_BASE}.v_player_onice_xg_game AS
    WITH onice AS (
      SELECT s.player_key, s.game_key, s.team_key,
        sh.xg_on_net, sh.is_sog, sh.is_slot,
        CASE WHEN sh.team_key=s.team_key THEN 1 ELSE 0 END AS is_for
      FROM {PROD_BASE}.fact_shift s
      JOIN {PROD_BASE}.fact_shot sh
        ON sh.game_key=s.game_key AND sh.period=s.period
        AND sh.period_time >= s.start_period_sec AND sh.period_time <= s.end_period_sec
      WHERE sh.strength_key='evenStrength:5:5' AND sh.is_shootout=0
    )
    SELECT d.calendar_date, o.game_key, o.player_key, p.player_name, pg.position,
           pg.team_key, tg.opponent_team_key,
           SUM(CASE WHEN o.is_for=1 THEN o.xg_on_net ELSE 0 END) AS xg_for_5v5,
           SUM(CASE WHEN o.is_for=0 THEN o.xg_on_net ELSE 0 END) AS xg_against_5v5,
           SUM(CASE WHEN o.is_for=1 THEN o.xg_on_net ELSE 0 END) - SUM(CASE WHEN o.is_for=0 THEN o.xg_on_net ELSE 0 END) AS xg_diff_5v5,
           SUM(CASE WHEN o.is_for=1 THEN 1 ELSE 0 END) AS shots_for_5v5,
           SUM(CASE WHEN o.is_for=0 THEN 1 ELSE 0 END) AS shots_against_5v5,
           SUM(CASE WHEN o.is_for=1 THEN o.is_sog ELSE 0 END) AS sog_for_5v5,
           SUM(CASE WHEN o.is_for=0 THEN o.is_sog ELSE 0 END) AS sog_against_5v5,
           SUM(CASE WHEN o.is_for=1 AND o.is_slot=1 THEN 1 ELSE 0 END) AS slot_shots_for_5v5,
           SUM(CASE WHEN o.is_for=0 AND o.is_slot=1 THEN 1 ELSE 0 END) AS slot_shots_against_5v5
    FROM onice o
    JOIN {PROD_BASE}.dim_player p ON p.player_key=o.player_key
    JOIN {PROD_BASE}.fact_player_game pg ON pg.game_key=o.game_key AND pg.player_key=o.player_key
    JOIN {PROD_BASE}.fact_team_game tg ON tg.game_key=o.game_key AND tg.team_key=o.team_key
    JOIN {PROD_BASE}.dim_date d ON d.date_key=pg.date_key
    GROUP BY d.calendar_date, o.game_key, o.player_key, p.player_name, pg.position, pg.team_key, tg.opponent_team_key
    """)
    spark.sql(f"""
    CREATE OR REPLACE VIEW {PROD_BASE}.v_player_event_season AS
    WITH event_season AS (
      SELECT player_key, player_name,
        ARRAY_JOIN(COLLECT_SET(position), '/') AS positions,
        COUNT(*) AS games,
        SUM(faceoff_wins) AS faceoff_wins, SUM(faceoff_losses) AS faceoff_losses,
        SUM(fo_wins_oz) AS fo_wins_oz, SUM(fo_wins_nz) AS fo_wins_nz, SUM(fo_wins_dz) AS fo_wins_dz,
        SUM(fo_losses_oz) AS fo_losses_oz, SUM(fo_losses_nz) AS fo_losses_nz, SUM(fo_losses_dz) AS fo_losses_dz,
        SUM(penalties_taken) AS penalties_taken, SUM(penalties_oz) AS penalties_oz, SUM(penalties_dz) AS penalties_dz,
        ARRAY_JOIN(ARRAY_DISTINCT(FLATTEN(COLLECT_LIST(CASE WHEN penalty_types IS NOT NULL THEN SPLIT(penalty_types, ', ') ELSE ARRAY() END))), ', ') AS penalty_types_all,
        SUM(oz_recoveries) AS oz_recoveries, SUM(dz_recoveries) AS dz_recoveries, SUM(nz_recoveries) AS nz_recoveries,
        SUM(contested_recoveries) AS contested_recoveries,
        SUM(oz_forecheck_wins) AS oz_forecheck_wins, SUM(dz_def_recoveries) AS dz_def_recoveries,
        SUM(controlled_exits_evt) AS controlled_exits_evt, SUM(zone_exits_ok) AS zone_exits_ok, SUM(zone_exits_failed) AS zone_exits_failed,
        SUM(breakouts_ok) AS breakouts_ok, SUM(breakouts_failed) AS breakouts_failed,
        SUM(pk_exits_ok) AS pk_exits_ok, SUM(pk_exits_failed) AS pk_exits_failed,
        SUM(controlled_entries_evt) AS controlled_entries_evt, SUM(carries) AS carries,
        SUM(end_to_end_rushes) AS end_to_end_rushes, SUM(pp_entries) AS pp_entries,
        SUM(scoring_chances_5v5) AS scoring_chances_5v5, SUM(scoring_chances_pp) AS scoring_chances_pp, SUM(scoring_chances_all) AS scoring_chances_all,
        SUM(xg_5v5) AS xg_5v5, SUM(xg_pp) AS xg_pp, SUM(sog_5v5) AS sog_5v5, SUM(sog_pp) AS sog_pp,
        SUM(puck_wins) AS puck_wins, SUM(puck_wins_with_shot) AS puck_wins_with_shot
      FROM {PROD_BASE}.v_player_event_game
      GROUP BY player_key, player_name
    ),
    onice_season AS (
      SELECT player_key,
        SUM(xg_for_5v5) AS xg_for_5v5, SUM(xg_against_5v5) AS xg_against_5v5,
        SUM(shots_for_5v5) AS shots_for_5v5, SUM(shots_against_5v5) AS shots_against_5v5,
        SUM(sog_for_5v5) AS sog_for_5v5, SUM(sog_against_5v5) AS sog_against_5v5,
        SUM(slot_shots_for_5v5) AS slot_shots_for_5v5, SUM(slot_shots_against_5v5) AS slot_shots_against_5v5
      FROM {PROD_BASE}.v_player_onice_xg_game
      GROUP BY player_key
    )
    SELECT e.*,
           CASE WHEN e.faceoff_wins+e.faceoff_losses>0
                THEN ROUND(e.faceoff_wins*100.0/(e.faceoff_wins+e.faceoff_losses),1)
                ELSE NULL END AS faceoff_pct,
           o.xg_for_5v5, o.xg_against_5v5,
           CASE WHEN o.xg_for_5v5 IS NOT NULL THEN o.xg_for_5v5-o.xg_against_5v5 END AS xg_diff_5v5,
           o.shots_for_5v5, o.shots_against_5v5,
           o.sog_for_5v5, o.sog_against_5v5,
           o.slot_shots_for_5v5, o.slot_shots_against_5v5
    FROM event_season e
    LEFT JOIN onice_season o ON o.player_key=e.player_key
    """)
    for view in ['v_skater_game','v_goalie_game','v_skater_season','v_goalie_season']:
        assert spark.table(PROD_INDIVIDUAL+'.'+view).count()==spark.table(WORK_INDIVIDUAL+'.'+view).count(), 'Vyavvikelse: '+view
    assert spark.table(PROD_BASE+'.v_team_game_safe').count()==spark.table(WORK_BASE+'.v_team_game_safe').count()
    save_run('SUCCESS',games=len(new_games),individual_games=len(game_ids),gold_build_id=RUN_ID,individual_build_id=RUN_ID)
except Exception:
    failures=restore_previous()
    save_run('RECOVERY_REQUIRED' if failures else 'FAILED_ROLLED_BACK',recovery_failures=failures)
    if failures:
        print('Återställningen behöver granskas. Logg:',RUN_DIR/'recovery.json')
    else:
        print('Tabellbytet misslyckades. Tidigare tabellversioner har återställts.')
    raise
finally:
    close_client()


## 11. Resultat
Skicka statusraden och kontrollsammanställningen från denna cell efter första körningen. Datakvalitetsavvikelser kan finnas även när bygget lyckas; de spärrade måtten ska fortsätta vara NULL.

In [0]:
assert build_status(PROD_BASE)==RUN_ID and build_status(PROD_INDIVIDUAL)==RUN_ID
assert json.loads((RUN_DIR/'status.json').read_text())['status']=='SUCCESS'
display(spark.createDataFrame([(RUN_ID,RUN_TIME,'SUCCESS',len(new_games),len(game_ids))],
    'run_id STRING, build_time_utc STRING, status STRING, loaded_games LONG, almtuna_games LONG'))
display(spark.sql(f"SELECT metric,status,COUNT(*) team_games FROM {PROD_INDIVIDUAL}.fact_metric_quality WHERE team_key='{TEAM_ID}' GROUP BY metric,status ORDER BY metric,status"))
display(spark.sql(f"SELECT game_id,error_count,warning_count,shots_usable,xg_quality_pass,toi_usable FROM {PROD_BASE}.dim_game_quality WHERE error_count>0 OR warning_count>0 ORDER BY game_id"))
print('SUCCESS — ladda om dashboarden:')
print('https://dbc-51cce3cf-cc96.cloud.databricks.com/sql/dashboardsv3/01f1b7433624165ca22cc76751112bd5')
print('Körningslogg:',RUN_DIR)
print('Nästa gång: öppna denna notebook och kör Run all.')

## Om körningen avbryts
- **Före tabellbytet:** tidigare analystabeller finns kvar. Rätta felet och kör Run all igen. Sparade endpoints återanvänds när källrevisionen stämmer.
- **Fångat fel under tabellbytet:** läs status i körningsloggen. `FAILED_ROLLED_BACK` betyder att återställningsanropen lyckades; `RECOVERY_REQUIRED` kräver granskning.
- **Klusterstopp under tabellbytet:** kör inte ett nytt bygge direkt. `before_versions.json` och `changed_tables.json` i utskriven körningsmapp anger föregående Delta-versioner och vilka tabeller som kan ha ändrats. Befintliga tabeller kan återställas med `RESTORE TABLE <tabell> TO VERSION AS OF <version>`; återställ byggstatustabeller sist. Nya tabeller har `null` som föregående version. Skicka statusfilen och felmeddelandet för att fortsätta från rätt punkt.

Manifester för gamla pilot-/täckningsbyggen beskriver fortfarande de äldre körningarna. Aktuell spelaranalys finns i `individual_v1`; aktuell passningstäckning finns i `passing_v1.fact_game_team_pass_coverage`. Råarkivet är bestående; de fem `refresh10_*`-schemana är arbetsytor som byggs om nästa gång.

### Tekniska referenser
[Databricks CREATE TABLE](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-ddl-create-table-using) · [Databricks RESTORE](https://docs.databricks.com/aws/en/sql/language-manual/delta-restore)
